# SheafPatternFusion Phase 3 (WP3.0 pivot-gate) - WP3.0c label-generation: forced-cyclic attack shard 00/4

Work package: **WP3.0c (feeds gate G2.6)**. Runs the full adversarial battery (A1/A2/A3, audit-identical budgets, non-shared oracles, pin-aware rebuild) on the forced cyclic stratum's engine-undecided x sheaf-RECOVERABLE rows (784 fleet-wide). Any confirmed false RECOVERABLE feeds D2-style kill accounting; every verdict becomes an attacker-derived label for the signal-validity AUC test.

Runtime: CPU-only (~2 cores). Expected wall time: **~2-4 h**. Everything is checkpointed to JSONL and resume-safe: re-running 'Run all' continues where the session stopped.

First run: the first cell installs the pinned numpy/scipy and HALTS with a message. Do Runtime > Restart session once (clears the preloaded binaries), then Runtime > Run all again; the install cell detects the pins and skips. The library is embedded in this notebook (generated from sheafpatternfusion source); no package install is needed.

In [ ]:
import importlib.metadata as md
import subprocess
import sys

WANT = {'numpy': '2.4.3', 'scipy': '1.17.1'}


def _ver(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return None


missing = {p: v for p, v in WANT.items() if _ver(p) != v}
if not missing:
    print('environment OK:', WANT)
else:
    print('installing pinned numpy/scipy (one-time per session) ...')
    res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                          'numpy==2.4.3', 'scipy==1.17.1'])
    if res.returncode != 0:
        raise RuntimeError('pip install failed; see log above')
    print()
    print('=' * 72)
    print('DEPENDENCIES INSTALLED. One manual step left:')
    print('  1) Runtime > Restart session ...   (clears the old numpy/scipy)')
    print('  2) Runtime > Run all               (this cell will skip)')
    print('=' * 72)
    raise SystemExit('restart required before importing numpy/scipy')


In [ ]:
import functools
import glob
import io
import json
import multiprocessing as mp
import os
import pathlib
import time
import urllib.request

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'


In [ ]:
# ==========================================================================
# EMBEDDED LIBRARY -- generated from src/sheafpatternfusion@
# (mdag_dgp.py, lp_ground_truth.py, enumerate_structures.py, gluing.py, battery.py, engine2.py, attackers.py, phase3_probe.py) by scripts/make_colab_phase3.py.
# Relative imports are stripped; this cell defines every symbol in the
# notebook namespace (concatenation order resolves all cross-module
# symbols, including MDAG). Do not edit by hand -- regenerate instead.
# ==========================================================================
from __future__ import annotations
"""m-graph data-generating process for discrete (binary) missing-data models.

Semantics follow Mohan-Pearl-Tian (2013) / Mohan-Pearl (2021): each binary
variable V_i has a structural mechanism P(v_i | parents); each missingness
indicator R_i has mechanism P(r_i | pa_G(R_i)) where pa_G(R_i) is a set of
VARIABLE indices and may include i itself (self-censoring MNAR edge).

Convention: r_i = 1 means V_i observed.

This module is ground-truth-side code only: it never uses sheaf machinery.
"""

import itertools
from dataclasses import dataclass, field

import numpy as np


@dataclass
class MDAG:
    n_vars: int
    var_parents: dict[int, tuple[int, ...]]
    r_parents: dict[int, tuple[int, ...]]  # may contain i itself (MNAR self-edge)
    var_cpt: dict[int, dict[tuple, float]] = field(default_factory=dict)
    r_cpt: dict[int, dict[tuple, float]] = field(default_factory=dict)

    def __post_init__(self):
        for i in range(self.n_vars):
            self.var_cpt.setdefault(i, {})
            self.r_cpt.setdefault(i, {})

    # ---------------- validation ----------------

    def validate_topological(self):
        """Variable mechanisms must depend on lower-indexed variables only;
        indicator mechanisms may depend on any variables (evaluation is
        conditional on the full v vector, so no ordering constraint applies)."""
        for i in range(self.n_vars):
            assert all(p < i for p in self.var_parents[i]), f"var parent order at {i}"

    def random_fill(self, rng: np.random.Generator):
        """Fill CPTs with uniform-random probabilities in [0.15, 0.85]."""
        for i in range(self.n_vars):
            pa = self.var_parents[i]
            keys = list(itertools.product(*[(0, 1)] * len(pa))) if pa else [()]
            self.var_cpt[i] = {k: rng.uniform(0.15, 0.85) for k in keys}
        for i in range(self.n_vars):
            pa = self.r_parents[i]
            keys = list(itertools.product(*[(0, 1)] * len(pa))) if pa else [()]
            self.r_cpt[i] = {k: rng.uniform(0.25, 0.75) for k in keys}

    # ---------------- exact laws ----------------

    def p_var(self, v: tuple[int, ...]) -> float:
        out = 1.0
        for i in range(self.n_vars):
            pa = tuple(v[p] for p in self.var_parents[i])
            p1 = self.var_cpt[i][pa]
            out *= p1 if v[i] == 1 else 1.0 - p1
        return out

    def p_r_given_v(self, r: tuple[int, ...], v: tuple[int, ...]) -> float:
        out = 1.0
        for i in range(self.n_vars):
            pa = tuple(v[p] for p in self.r_parents[i])  # includes i if self-edge
            q = self.r_cpt[i][pa]
            out *= q if r[i] == 1 else 1.0 - q
        return out

    def joint_table(self) -> dict[tuple[tuple, tuple], float]:
        """P(v, r) over all cells; entries may be zero via mechanisms."""
        out = {}
        for v in itertools.product((0, 1), repeat=self.n_vars):
            pv = self.p_var(v)
            if pv == 0.0:
                continue
            for r in itertools.product((0, 1), repeat=self.n_vars):
                out[(v, r)] = pv * self.p_r_given_v(r, v)
        return out

    def observed_laws(self, jt: dict | None = None) -> dict[tuple, dict[tuple, float]]:
        """q_r(o) = P(V_O=o | R=r) for every realized pattern r."""
        jt = self.joint_table() if jt is None else jt
        num: dict[tuple, dict[tuple, float]] = {}
        den: dict[tuple, float] = {}
        for (v, r), p in jt.items():
            o = tuple(v[i] for i in range(self.n_vars) if r[i] == 1)
            num.setdefault(r, {}).setdefault(o, 0.0)
            num[r][o] += p
            den[r] = den.get(r, 0.0) + p
        return {
            r: {o: c / den[r] for o, c in cells.items()}
            for r, cells in num.items()
            if den.get(r, 0.0) > 0.0
        }

    def realized_patterns(self, tol: float = 0.0, jt: dict | None = None) -> list[tuple]:
        jt = self.joint_table() if jt is None else jt
        den: dict[tuple, float] = {}
        for (v, r), p in jt.items():
            den[r] = den.get(r, 0.0) + p
        return sorted([r for r, p in den.items() if p > tol])

    # ---------------- sampling ----------------

    def sample(self, n: int, seed: int) -> tuple[np.ndarray, np.ndarray]:
        """Draw n individuals; returns (V[n,n], R[n,n]) with full truth."""
        rng = np.random.default_rng(seed)
        V = np.zeros((n, self.n_vars), dtype=np.int64)
        for row in range(n):
            v = []
            for i in range(self.n_vars):
                pa = tuple(v[p] for p in self.var_parents[i])
                v.append(int(rng.random() < self.var_cpt[i][pa]))
            V[row] = v
        R = np.zeros((n, self.n_vars), dtype=np.int64)
        for row in range(n):
            v = list(V[row])
            r = []
            for i in range(self.n_vars):
                pa = tuple(v[p] for p in self.r_parents[i])
                r.append(int(rng.random() < self.r_cpt[i][pa]))
            R[row] = r
        return V, R

    @staticmethod
    def empirical_observed_laws(V: np.ndarray, R: np.ndarray) -> dict[tuple, dict[tuple, float]]:
        """Empirical pattern-conditional laws from a masked dataset."""
        n, d = V.shape
        out: dict[tuple, dict[tuple, float]] = {}
        cnt: dict[tuple, int] = {}
        for k in range(n):
            r = tuple(int(x) for x in R[k])
            o = tuple(int(x) for x in V[k][np.array(r, dtype=bool)])
            out.setdefault(r, {}).setdefault(o, 0.0)
            out[r][o] += 1.0
            cnt[r] = cnt.get(r, 0) + 1
        return {r: {o: c / cnt[r] for o, c in cells.items()} for r, cells in out.items()}

"""Ground-truth recoverability engine for small binary missing-data instances.

Three instruments, combined with an explicit precedence rule (see `decide`):

1. Assumption-free LP relaxation (`lp_range`): linear program over full-table
   cells t[v, r] whose induced observed conditionals match reference observed
   laws exactly. Any two feasible points are valid joint laws of (V, R)
   (mechanism assumptions dropped), so a functional that varies over this
   polytope is CERTIFIED unrecoverable under any submodel (sound one way).
   Witness tables returned.

2. Model-aware witness search (`model_witness_search`): constrained nonlinear
   optimization over CPT parameters theta with equality constraints
   F(theta) == F(theta_ref) (identical observed laws under the m-graph
   factorization). A pair with distance < 1e-9 and |dphi| > tol certifies
   unrecoverability UNDER THE MODEL numerically.

3. Identification-formula checks (`IDENTITY_FORMULAS`): closed-form estimators
   that equal the target under the instance's structure class; verified
   numerically against the true generating model at machine precision.

Verdict precedence in `decide`: formula-pass -> RECOVERABLE (formula-certified);
else model witness -> UNRECOVERABLE; else LP width -> UNRECOVERABLE_RELAXED;
else UNDETERMINED. Positive bank instances carry formulas; negatives rely on
witnesses, so no bank instance lands in UNDETERMINED.
"""

import copy
import itertools

import numpy as np
from scipy.optimize import least_squares, linprog, minimize



# --------------------------------------------------------------------------
# packing / unpacking CPT parameters
# --------------------------------------------------------------------------

def param_spec(inst: MDAG):
    spec = []
    for i in range(inst.n_vars):
        for k in inst.var_cpt[i]:
            spec.append(("var", i, k))
    for i in range(inst.n_vars):
        for k in inst.r_cpt[i]:
            spec.append(("r", i, k))
    return spec


def pack(inst: MDAG) -> np.ndarray:
    out = np.zeros(len(param_spec(inst)))
    for idx, (kind, i, k) in enumerate(param_spec(inst)):
        out[idx] = inst.var_cpt[i][k] if kind == "var" else inst.r_cpt[i][k]
    return out


def unpack(inst: MDAG, theta: np.ndarray) -> MDAG:
    new = copy.deepcopy(inst)
    for idx, (kind, i, k) in enumerate(param_spec(inst)):
        if kind == "var":
            new.var_cpt[i][k] = float(theta[idx])
        else:
            new.r_cpt[i][k] = float(theta[idx])
    return new


# --------------------------------------------------------------------------
# targets
# --------------------------------------------------------------------------

def param_bounds(inst: MDAG):
    """Per-parameter (lo, hi) arrays. Entries pinned by exact mechanism values
    (e.g., an always-observed indicator with p=1.0) are frozen at their value
    so search spaces remain consistent with the reference model."""
    lo = np.zeros(len(param_spec(inst)))
    hi = np.ones(len(param_spec(inst)))
    for idx, (kind, i, k) in enumerate(param_spec(inst)):
        table = inst.var_cpt if kind == "var" else inst.r_cpt
        val = table[i][k]
        if val in (0.0, 1.0):
            lo[idx] = hi[idx] = val
        else:
            lo[idx] = 0.03 if kind == "var" else 0.05
            hi[idx] = 0.97 if kind == "var" else 0.95
    return lo, hi


def target_value_phi(inst: MDAG, target) -> float:
    """Evaluate the target functional on the model's variable law P(v)."""
    kind = target[0]
    if kind == "cond":
        _, y_i, y_val, x_idx, x_val = target
        num = den = 0.0
        for v in itertools.product((0, 1), repeat=inst.n_vars):
            pv = inst.p_var(v)
            if tuple(v[i] for i in x_idx) == tuple(x_val):
                den += pv
                if v[y_i] == y_val:
                    num += pv
        assert den > 0
        return num / den
    tot = 0.0
    for v in itertools.product((0, 1), repeat=inst.n_vars):
        pv = inst.p_var(v)
        if kind == "mean":
            val = float(v[target[1]])
        elif kind == "cell":
            val = 1.0 if tuple(v) == tuple(target[1]) else 0.0
        else:
            raise ValueError(kind)
        tot += pv * val
    return tot


def _lp_coeffs(inst: MDAG, target) -> dict[tuple, float]:
    kind = target[0]
    vals = {}
    for v in itertools.product((0, 1), repeat=inst.n_vars):
        if kind == "mean":
            vals[v] = float(v[target[1]])
        elif kind == "cell":
            vals[v] = 1.0 if tuple(v) == tuple(target[1]) else 0.0
        else:
            raise ValueError("LP supports 'mean' and 'cell' targets only")
    return vals


# --------------------------------------------------------------------------
# helpers on observed laws
# --------------------------------------------------------------------------

def pattern_probabilities(inst: MDAG) -> dict[tuple, float]:
    jt = inst.joint_table()
    out: dict[tuple, float] = {}
    for (v, r), p in jt.items():
        out[r] = out.get(r, 0.0) + p
    return out


def marginalize(q_r: dict[tuple, float], r: tuple[int, ...], keep: tuple[int, ...]) -> dict[tuple, float]:
    """Marginal of pattern-conditional table onto original indices `keep`."""
    def pos(i: int) -> int:
        return sum(1 for j in range(len(r)) if r[j] == 1 and j < i)

    assert all(r[i] == 1 for i in keep)
    out: dict[tuple, float] = {}
    for o, c in q_r.items():
        key = tuple(o[pos(i)] for i in keep)
        out[key] = out.get(key, 0.0) + c
    return out


def cond_from_full_stratum(q_full: dict[tuple, float], n_vars: int,
                           y_i: int, y_val: int, x_idx: tuple[int, ...], x_val: tuple[int, ...]) -> float:
    num = den = 0.0
    for o, c in q_full.items():
        if tuple(o[i] for i in x_idx) == tuple(x_val):
            den += c
            if o[y_i] == y_val:
                num += c
    return num / den


# --------------------------------------------------------------------------
# instrument 1: assumption-free LP relaxation
# --------------------------------------------------------------------------

def lp_range(inst: MDAG, q_hat: dict[tuple, dict[tuple, float]], target):
    """Max/min of target over ALL joint tables matching observed margins.

    Variables: x = [t cells | c_r scales]. Constraints: sum(t)=1 and, for each
    realized pattern r and config o: sum_{v: v_O=o} t[v,r] = c_r * q_r(o).
    """
    patterns = sorted(q_hat.keys())
    cells = list(itertools.product(itertools.product((0, 1), repeat=inst.n_vars), patterns))
    T, Rpats = len(cells), len(patterns)
    phi_vals = _lp_coeffs(inst, target)

    rows, rhs = [], []
    rows.append(np.ones(T + Rpats))
    rhs.append(1.0)
    for j, r in enumerate(patterns):
        Oidx = [i for i in range(inst.n_vars) if r[i] == 1]
        for o in itertools.product((0, 1), repeat=len(Oidx)):
            row = np.zeros(T + Rpats)
            for ci, (v, rr) in enumerate(cells):
                if rr == r and tuple(v[i] for i in Oidx) == tuple(o):
                    row[ci] = 1.0
            row[T + j] = -q_hat[r][tuple(o)]
            rows.append(row)
            rhs.append(0.0)

    A_eq, b_eq = np.array(rows), np.array(rhs)
    bounds = [(0.0, 1.0)] * T + [(0.0, None)] * Rpats

    def solve(direction):
        c_obj = np.array([phi_vals[v] for v, _ in cells] + [0.0] * Rpats)
        res = linprog(direction * c_obj, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
        assert res.status == 0, f"LP failed: {res.message}"
        return res.fun * direction, res.x

    lo, xlo = solve(1.0)
    hi, xhi = solve(-1.0)
    return {
        "lo": lo,
        "hi": hi,
        "width": hi - lo,
        "t_min": {cells[k]: float(xlo[k]) for k in range(T)},
        "t_max": {cells[k]: float(xhi[k]) for k in range(T)},
    }


# --------------------------------------------------------------------------
# instrument 2: model-aware witness search
# --------------------------------------------------------------------------

def observed_vector(inst: MDAG, patterns) -> tuple[np.ndarray, list]:
    """Observed-data fingerprint: per-pattern CONDITIONAL laws AND the pattern
    probabilities P(R=r). Both are observable, so completions must match both."""
    jt = inst.joint_table()
    q = inst.observed_laws(jt)
    pp = {}
    for (v, r), p in jt.items():
        pp[r] = pp.get(r, 0.0) + p
    keys, vals = [], []
    for r in patterns:
        keys.append(("pat", r))
        vals.append(pp.get(r, 0.0))
        Oidx = [i for i in range(inst.n_vars) if r[i] == 1]
        qr = q.get(r, {})
        for o in itertools.product((0, 1), repeat=len(Oidx)):
            keys.append((r, o))
            vals.append(qr.get(tuple(o), 0.0))
    return np.array(vals), keys


def _jacobian(inst, theta, patterns, f_ref, eps=1e-6):
    spec_n = len(param_spec(inst))
    J = np.zeros((len(f_ref), spec_n))
    for j in range(spec_n):
        tp = theta.copy(); tp[j] += eps
        tm = theta.copy(); tm[j] -= eps
        fp, _ = observed_vector(unpack(inst, tp), patterns)
        fm, _ = observed_vector(unpack(inst, tm), patterns)
        J[:, j] = (fp - fm) / (2 * eps)
    return J


def manifold_walk(inst, theta_ref, target, n_seeds: int = 12, steps: int = 60,
                  step_size: float = 0.02, seed: int = 0,
                  dist_tol: float = 1e-9):
    """Walk the feasible manifold {theta : F(theta) = F(theta_ref)} using
    null-space predictor + first-order corrector, maximizing |dphi|. Much more
    reliable than generic SLSQP for tiny overidentified systems."""
    rng = np.random.default_rng(seed)
    ref_inst = unpack(inst, theta_ref)
    patterns = ref_inst.realized_patterns(jt=ref_inst.joint_table())
    f_ref, _ = observed_vector(ref_inst, patterns)
    phi_ref = target_value_phi(ref_inst, target)
    bounds_lo = np.array([0.03 if k == "var" else 0.05 for k, _, _ in param_spec(inst)])
    bounds_hi = np.array([0.97 if k == "var" else 0.95 for k, _, _ in param_spec(inst)])

    best = {"delta_phi": 0.0, "dist": np.inf, "success": False,
            "theta_pair": None, "phi_values": None}

    def record(x):
        nonlocal best
        m = unpack(inst, x)
        f_new, _ = observed_vector(m, patterns)
        dist = float(np.max(np.abs(f_new - f_ref)))
        dphi = abs(target_value_phi(m, target) - phi_ref)
        if dist < dist_tol and dphi > best["delta_phi"]:
            best = {"delta_phi": float(dphi), "dist": dist,
                    "success": bool(dphi > 1e-4),
                    "theta_pair": (theta_ref.copy(), x.copy()),
                    "phi_values": (float(phi_ref), float(target_value_phi(m, target)))}

    # estimate Jacobian once at reference point
    J = _jacobian(inst, theta_ref, patterns, f_ref)
    U, S, Vt = np.linalg.svd(J, full_matrices=True)
    r = int(np.sum(S > 1e-8))
    Null = Vt[r:].T if r < Vt.shape[1] else np.zeros((len(theta_ref), 0))
    Jp = np.linalg.pinv(J)

    for k in range(n_seeds):
        d = Null[:, k % Null.shape[1]] if Null.size else None
        if d is None:
            break
        sign = 1.0 if rng.random() < 0.5 else -1.0
        x = theta_ref.copy()
        for _ in range(steps):
            x = x + sign * step_size * d / max(np.linalg.norm(d), 1e-12)
            x = np.clip(x, bounds_lo, bounds_hi)
            # first-order corrector back onto the manifold
            for _ in range(3):
                fx, _ = observed_vector(unpack(inst, x), patterns)
                x = x - Jp @ (fx - f_ref)
                x = np.clip(x, bounds_lo, bounds_hi)
            fx, _ = observed_vector(unpack(inst, x), patterns)
            if np.max(np.abs(fx - f_ref)) > 1e-7:
                break
            record(x)
            # refresh direction along curved manifold occasionally
            Jx = _jacobian(inst, x, patterns, f_ref)
            _, Sx, Vtx = np.linalg.svd(Jx, full_matrices=True)
            rx = int(np.sum(Sx > 1e-8))
            if rx < Vtx.shape[1]:
                Null = np.hstack([Null, Vtx[rx:].T])
    return best


def root_jump_search(inst: MDAG, theta_ref: np.ndarray, target,
                     n_starts: int = 40, seed: int = 0,
                     dist_tol: float = 1e-8):
    """Find distinct factorized models sharing the observed law of theta_ref by
    multistart least-squares root finding on F(theta) - F(theta_ref). Tiny
    overidentified moment systems typically admit many isolated roots; jumping
    between them is the most effective witness strategy at Phase-1 sizes."""
    rng = np.random.default_rng(seed)
    ref_inst = unpack(inst, theta_ref)
    patterns = ref_inst.realized_patterns(jt=ref_inst.joint_table())
    f_ref, _ = observed_vector(ref_inst, patterns)
    phi_ref = target_value_phi(ref_inst, target)
    lo, hi = param_bounds(inst)
    free = np.where(hi - lo > 0)[0]
    base = theta_ref.copy()

    def expand(x_free):
        th = base.copy()
        th[free] = x_free
        return th

    best = {"delta_phi": 0.0, "dist": np.inf, "success": False,
            "theta_pair": None, "phi_values": None}
    if len(free) == 0:
        return best
    for _ in range(n_starts):
        span = hi[free] - lo[free]
        x0 = lo[free] + 0.02 * span + rng.random(len(free)) * (0.96 * span)
        res = least_squares(
            lambda xf: observed_vector(unpack(inst, expand(xf)), patterns)[0] - f_ref,
            x0, bounds=(lo[free], hi[free]), xtol=1e-15, ftol=1e-15, gtol=1e-15)
        if np.max(np.abs(res.fun)) >= dist_tol:
            continue
        m = unpack(inst, expand(res.x))
        dphi = abs(target_value_phi(m, target) - phi_ref)
        if dphi > best["delta_phi"]:
            best = {"delta_phi": float(dphi), "dist": float(np.max(np.abs(res.fun))),
                    "success": bool(dphi > 1e-4),
                    "theta_pair": (theta_ref.copy(), expand(res.x).copy()),
                    "phi_values": (float(phi_ref), float(target_value_phi(m, target)))}
            if best["delta_phi"] > 0.5:
                break
    return best


def model_witness_search(inst: MDAG, theta_ref: np.ndarray, target,
                         n_starts: int = 24, seed: int = 0,
                         dist_tol: float = 1e-9, phi_tol: float = 1e-4):
    rng = np.random.default_rng(seed)
    spec_n = len(param_spec(inst))
    ref_inst = unpack(inst, theta_ref)
    patterns = ref_inst.realized_patterns(jt=ref_inst.joint_table())
    f_ref, _ = observed_vector(ref_inst, patterns)
    phi_ref = target_value_phi(ref_inst, target)
    lo_b, hi_b = param_bounds(inst)
    bounds = list(zip(lo_b, hi_b))

    best = {"delta_phi": 0.0, "dist": np.inf, "success": False,
            "theta_pair": None, "phi_values": None}

    cons = [{"type": "eq", "fun": lambda x: observed_vector(unpack(inst, x), patterns)[0] - f_ref}]

    lo_b, hi_b = param_bounds(inst)
    span_b = hi_b - lo_b

    def in_bounds(x):
        return np.where(span_b > 0,
                        np.clip(x, lo_b + 1e-3 * span_b, hi_b - 1e-3 * span_b),
                        lo_b)

    starts = [in_bounds(theta_ref + rng.normal(0, 0.06, size=spec_n))
              for _ in range(n_starts // 2)]
    starts += [in_bounds(lo_b + rng.random(spec_n) * span_b)
               for _ in range(n_starts - len(starts))]

    for x0 in starts:
        for sign in (+1.0, -1.0):

            def obj(x, sign=sign):
                m = unpack(inst, x)
                gap = np.max(np.abs(observed_vector(m, patterns)[0] - f_ref))
                pen = 1e4 * max(gap - 1e-12, 0.0)
                return sign * (target_value_phi(m, target) - phi_ref) + pen

            res = minimize(obj, x0, method="SLSQP", bounds=bounds, constraints=cons,
                           options={"maxiter": 300, "ftol": 1e-12})
            if not np.all(np.isfinite(res.x)):
                continue
            m = unpack(inst, res.x)
            dist = float(np.max(np.abs(observed_vector(m, patterns)[0] - f_ref)))
            dphi = abs(target_value_phi(m, target) - phi_ref)
            if dist < dist_tol and dphi > best["delta_phi"]:
                best = {"delta_phi": float(dphi), "dist": dist,
                        "success": bool(dphi > phi_tol),
                        "theta_pair": (theta_ref.copy(), res.x.copy()),
                        "phi_values": (float(phi_ref), float(target_value_phi(m, target)))}
            if best["delta_phi"] > 0.5:
                return best
    return best


# --------------------------------------------------------------------------
# instrument 3: identification formulas
# --------------------------------------------------------------------------

IDENTITY_FORMULAS = {}


def register(name):
    def deco(fn):
        IDENTITY_FORMULAS[name] = fn
        return fn

    return deco


@register("direct_full_pattern")
def f_direct(inst, q, target, aux):
    """Read mean/cell off a fully observed realized pattern (MCAR sanity)."""
    r_full = tuple([1] * inst.n_vars)
    cells = q[r_full]
    if target[0] == "cell":
        return cells[tuple(target[1])]
    tot = 0.0
    for o, c in cells.items():
        tot += c * o[target[1]]
    return tot


@register("mcar_cross_pattern_agreement")
def f_mcar_agree(inst, q, target, aux):
    """MCAR: every pattern observing the needed coordinates sees the same law."""
    if target[0] == "cell":
        ests = []
        for r, cells in q.items():
            need = tuple(i for i in range(inst.n_vars) if target[1][i] is not None)
            mg = marginalize(cells, r, need)
            key = tuple(target[1][i] for i in need)
            if key in mg:
                ests.append(mg[key])
        assert max(ests) - min(ests) < 1e-9, "MCAR cross-pattern disagreement"
        return float(np.mean(ests))
    j = target[1]
    ests = []
    for r, cells in q.items():
        if r[j] == 1:
            mg = marginalize(cells, r, (j,))
            ests.append(sum(k[0] * c for k, c in mg.items()))
    assert max(ests) - min(ests) < 1e-9, "MCAR cross-pattern disagreement"
    return float(np.mean(ests))


@register("mar_cond_stratum")
def f_mar_cond(inst, q, target, aux):
    """P(Y=y|X=x) inside the fully observed stratum (MAR identification)."""
    _, y_i, y_val, x_idx, x_val = target
    r_full = tuple([1] * inst.n_vars)
    return cond_from_full_stratum(q[r_full], inst.n_vars, y_i, y_val, x_idx, x_val)


@register("anchor_direct")
def f_anchor(inst, q, target, aux):
    """Mean of an always-observed variable pooled over patterns observing it."""
    j = target[1]
    num = den = 0.0
    pp = aux["pattern_prob"]
    for r, cells in q.items():
        if r[j] != 1:
            continue
        w = pp[r]
        mg = marginalize(cells, r, (j,))
        num += w * sum(k[0] * c for k, c in mg.items())
        den += w * sum(mg.values())
    return num / den


@register("mar_mean_iterated")
def f_mar_mean(inst, q, target, aux):
    """E[V_j] = sum_x P(x) P(V_j=1 | X=x); MAR stratum conditional plus pooled
    marginal of the conditioning block (all its members always observed)."""
    j = target[1]
    r_full = tuple([1] * inst.n_vars)
    always = [i for i in range(inst.n_vars)
              if all(r[i] == 1 for r in q.keys())]
    assert j in always or True  # j itself may be the partially observed one
    others = tuple(i for i in always if i != j)
    pxx: dict[tuple, float] = {}
    totw = 0.0
    pp = aux["pattern_prob"]
    for r, cells in q.items():
        if all(r[i] == 1 for i in others):
            w = pp[r]
            totw += w
            mg = marginalize(cells, r, others)
            for k, c in mg.items():
                pxx[k] = pxx.get(k, 0.0) + w * c
    pxx = {k: c / totw for k, c in pxx.items()}
    ex = 0.0
    for xx, wx in pxx.items():
        ex += wx * cond_from_full_stratum(q[r_full], inst.n_vars, j, 1, others, xx)
    return ex


@register("mar_joint_product")
def f_mar_joint(inst, q, target, aux):
    """Two-variable joint P(v1,v2) = P(v1) * P(v2|v1): pooled always-observed
    marginal of V1 times the MAR fully-observed-stratum conditional."""
    cell = tuple(target[1])
    r_full = tuple([1] * inst.n_vars)
    pxx: dict[tuple, float] = {}
    totw = 0.0
    pp = aux["pattern_prob"]
    for r, cells in q.items():
        if r[0] == 1:
            w = pp[r]
            totw += w
            mg = marginalize(cells, r, (0,))
            for k, c in mg.items():
                pxx[k] = pxx.get(k, 0.0) + w * c
    pxx = {k: c / totw for k, c in pxx.items()}
    return pxx[(cell[0],)] * cond_from_full_stratum(
        q[r_full], inst.n_vars, 1, cell[1], (0,), (cell[0],))


@register("cond_sel_stratum")
def f_cond_sel(inst, q, target, aux):
    """P(Y=y|X=x) from the selected stratum (selection depends on X alone)."""
    _, y_i, y_val, x_idx, x_val = target
    r_full = tuple([1] * inst.n_vars)
    return cond_from_full_stratum(q[r_full], inst.n_vars, y_i, y_val, x_idx, x_val)


# --------------------------------------------------------------------------
# decision procedure
# --------------------------------------------------------------------------

def decide(inst: MDAG, theta_true: np.ndarray, target, formula: str | None,
           seed: int = 0, lp_width_tol: float = 1e-3):
    m_true = unpack(inst, theta_true)
    jt = m_true.joint_table()
    q = m_true.observed_laws(jt)
    aux = {"pattern_prob": pattern_probabilities(m_true)}
    true_phi = target_value_phi(m_true, target)
    out = {"target": list(target), "true_value": true_phi}

    if formula is not None:
        est = IDENTITY_FORMULAS[formula](m_true, q, target, aux)
        out["formula_estimate"] = est
        if abs(est - true_phi) <= 1e-8 * max(1.0, abs(true_phi)):
            out.update(verdict="RECOVERABLE", evidence=f"formula:{formula}")
            return out
        out["formula_gap"] = abs(est - true_phi)

    wit = model_witness_search(inst, theta_true, target, seed=seed)
    out["witness"] = {k: wit[k] for k in ("delta_phi", "dist", "success", "phi_values")}
    if not wit["success"]:
        walk = root_jump_search(inst, theta_true, target, seed=seed + 1)
        out["root_jump"] = {k: walk[k] for k in ("delta_phi", "dist", "success")}
        if walk["success"]:
            wit = walk
            out["witness"] = {k: walk[k] for k in ("delta_phi", "dist", "success", "phi_values")}
    if wit["success"]:
        out.update(verdict="UNRECOVERABLE",
                   evidence=f"model_witness dphi={wit['delta_phi']:.4f} dist={wit['dist']:.1e}")
        return out

    if target[0] in ("mean", "cell"):
        lp = lp_range(inst, q, target)
        out["lp"] = {"width": lp["width"], "lo": lp["lo"], "hi": lp["hi"]}
        if lp["width"] > lp_width_tol:
            # Assumption-free variation only: the target varies over tables
            # matching the observed margins WITHOUT mechanism constraints.
            # This is evidence of fragility, NOT a model-valid unrecoverability
            # certificate (the model class is smaller than the relaxation).
            out.update(verdict="VARIABLE_UNCONSTRAINED_ONLY",
                       evidence=f"lp_width={lp['width']:.4f}")
            return out

    out.update(verdict="UNDETERMINED", evidence="no certificate either way")
    return out

"""Phase 2 enumeration infrastructure (WP2.1).

Structure space: m-graphs on binary variables with topologically ordered
variable mechanisms (V_i may only depend on lower-indexed variables) and
missingness mechanisms P(R_i | pa) where pa may include any variable plus
R_i's own variable (self-censoring MNAR edge). A STRUCTURE is
(var_parents, r_parents); an INSTANCE adds seeded CPT parameters and targets.

Everything here is ground-truth-side or structure-side bookkeeping; no sheaf
machinery is used for verdicts (engine side stays assumption-clean).
"""

import copy
import itertools
from dataclasses import dataclass

import numpy as np



# --------------------------------------------------------------------------
# structure generation
# --------------------------------------------------------------------------

def _subsets(pool: tuple[int, ...]) -> list[tuple[int, ...]]:
    return [tuple(x for x in pool if (m >> pool.index(x)) & 1)
            for m in range(2 ** len(pool))]


def var_dags(n: int) -> list[dict[int, tuple[int, ...]]]:
    """All parent assignments respecting V_i depends on lower indices only."""
    blocks = [_subsets(tuple(range(i))) for i in range(n)]
    return [{i: c for i, c in enumerate(choice)}
            for choice in itertools.product(*blocks)]


def r_mechanisms(n: int) -> list[tuple[tuple[int, ...], ...]]:
    """All parent tuples (pa(R_0), ..., pa(R_{n-1})); pa(R_i) is a subset of
    the variable set, and containing i itself is the self-censoring edge."""
    block = _subsets(tuple(range(n)))
    return [tuple(choice) for choice in itertools.product(block, repeat=n)]


def all_structures(n: int) -> list[tuple[dict[int, tuple[int, ...]], tuple]]:
    vds = var_dags(n)
    rms = r_mechanisms(n)
    return [(vd, rm) for vd in vds for rm in rms]


def instantiate(structure, seed: int, fixed_cpt: list[dict] | None = None) -> MDAG:
    vp, rp = structure
    inst = MDAG(n_vars=len(vp), var_parents=dict(vp),
                r_parents={i: tuple(p) for i, p in enumerate(rp)})
    inst.validate_topological()
    rng = np.random.default_rng(seed)
    inst.random_fill(rng)
    for fx in fixed_cpt or []:
        table = inst.r_cpt if fx["kind"] == "r" else inst.var_cpt
        table[fx["node"]][tuple(fx["parents"])] = float(fx["p"])
    return inst


# --------------------------------------------------------------------------
# mechanism classification (per drawn instance; depends on realized patterns)
# --------------------------------------------------------------------------

def classify(inst: MDAG) -> dict:
    jt = inst.joint_table()
    patterns = inst.realized_patterns(jt=jt)
    always = [i for i in range(inst.n_vars) if all(r[i] == 1 for r in patterns)]
    never = [i for i in range(inst.n_vars) if all(r[i] == 0 for r in patterns)]
    has_self = any(i in inst.r_parents[i] for i in range(inst.n_vars))
    nonempty_pa = any(len(inst.r_parents[i]) > 0 for i in range(inst.n_vars))
    is_mcar = not nonempty_pa
    pa_all_observed = all(all(p in always for p in inst.r_parents[i])
                          for i in range(inst.n_vars))
    is_mar = (not is_mcar) and pa_all_observed
    if is_mcar:
        cls = "MCAR"
    elif is_mar:
        cls = "MAR"
    elif has_self:
        cls = "MNAR_self"
    else:
        cls = "MNAR_other"
    return {
        "mechanism_class": cls,
        "is_mcar": is_mcar,
        "is_mar": is_mar,
        "has_self_edge": has_self,
        "always_observed": tuple(always),
        "never_observed": tuple(never),
        "n_realized_patterns": len(patterns),
    }


def poset_shape(patterns: list[tuple[int, ...]]) -> str:
    """Coarse shape label keyed on the overlap hypergraph (the structure that
    governs gluing): chain / acyclic (Berge) / cyclic."""
    ps = sorted(patterns)
    if all(all(a[i] <= b[i] for i in range(len(a))) for a, b in zip(ps, ps[1:])):
        return "chain"
    sets = [frozenset(i for i in range(len(p)) if p[i] == 1) for p in ps]
    return "acyclic" if graham_acyclic(sets) else "cyclic"


def graham_acyclic(observed_sets: list[frozenset[int]]) -> bool:
    """Graham's algorithm: Berge-acyclicity of the overlap hypergraph. True
    posets (running-intersection property) are where pairwise gluing suffices
    classically; this is the WP2.3 readout key."""
    edges = [set(s) for s in observed_sets if s]
    while len(edges) > 1:
        # find an edge whose intersection with the union of others equals its
        # intersection with ONE other edge (a "leaf")
        found = False
        for k, e in enumerate(edges):
            others = [o for j, o in enumerate(edges) if j != k]
            rest = set().union(*others)
            shared_others = [(frozenset(e & o), o) for o in others if e & o]
            if not shared_others:
                edges.pop(k)
                found = True
                break
            # removable if all its elements shared with others live in one other edge
            touching = frozenset(e & rest)
            if any(touching <= o for _, o in shared_others):
                edges.pop(k)
                found = True
                break
            if not (e & rest):
                edges.pop(k)
                found = True
                break
        if not found:
            return False
    return True


# --------------------------------------------------------------------------
# implied slice CIs (structural, discovered across random parameter draws)
# --------------------------------------------------------------------------

def _ci_candidates(observed: tuple[int, ...]) -> list[tuple[tuple, tuple, tuple]]:
    idx = [i for i in range(len(observed)) if observed[i] == 1]
    out = []
    k = len(idx)
    for mask in range(3 ** k):
        assign = []
        m = mask
        for _ in range(k):
            assign.append(m % 3)
            m //= 3
        x = tuple(idx[j] for j in range(k) if assign[j] == 0)
        y = tuple(idx[j] for j in range(k) if assign[j] == 1)
        z = tuple(idx[j] for j in range(k) if assign[j] == 2)
        if x and y:
            out.append((x, y, z))
    return out


def discover_slice_cis(inst: MDAG, n_draws: int = 24, tol: float = 1e-7,
                       seed: int = 12345) -> dict[tuple, list[tuple]]:
    """CIs (X,Y,Z) that hold on pattern r's normalized slice for EVERY random
    parameter draw: structural consequences of the m-graph, mechanically
    verified. Returns {pattern: [(X,Y,Z), ...]}."""
    def holds(table_norm: dict[tuple, float], observed, x, y, z) -> bool:
        def pos(i):
            return sum(1 for j in range(len(observed)) if observed[j] == 1 and j < i)

        px, py, pz = [pos(i) for i in x], [pos(i) for i in y], [pos(i) for i in z]
        joint: dict = {}
        for o, c in table_norm.items():
            key = (tuple(o[i] for i in px), tuple(o[i] for i in py),
                   tuple(o[i] for i in pz))
            joint[key] = joint.get(key, 0.0) + c
        pzm = {}
        for (kx, ky, kz), c in joint.items():
            pzm[kz] = pzm.get(kz, 0.0) + c
        for kz, cz in pzm.items():
            mx, my, mxy = {}, {}, {}
            for (kx2, ky2, kz2), c2 in joint.items():
                if kz2 != kz:
                    continue
                mx[kx2] = mx.get(kx2, 0.0) + c2
                my[ky2] = my.get(ky2, 0.0) + c2
                mxy[(kx2, ky2)] = c2
            for (kx2, ky2), cxy in mxy.items():
                if abs(cxy - mx[kx2] * my[ky2] / cz) > tol * max(1e-8, cz):
                    return False
        return True

    patterns = inst.realized_patterns(jt=inst.joint_table())
    cands = {r: _ci_candidates(r) for r in patterns}
    alive = {r: list(cands[r]) for r in patterns}
    rng = np.random.default_rng(seed)
    for _ in range(n_draws):
        m = copy.deepcopy(inst)
        for i in range(m.n_vars):
            pa = m.var_parents[i]
            keys = list(itertools.product(*[(0, 1)] * len(pa))) if pa else [()]
            m.var_cpt[i] = {k: float(rng.uniform(0.15, 0.85)) for k in keys}
        for i in range(m.n_vars):
            pa = m.r_parents[i]
            keys = list(itertools.product(*[(0, 1)] * len(pa))) if pa else [()]
            m.r_cpt[i] = {k: float(rng.uniform(0.25, 0.75)) for k in keys}
        q = m.observed_laws()
        for r in patterns:
            if r not in q:
                alive[r] = []
                continue
            tab = q[r]
            alive[r] = [c for c in alive[r] if holds(tab, r, *c)]
        if all(not v for v in alive.values()):
            break
    return {r: alive[r] for r in patterns}


# --------------------------------------------------------------------------
# targets and conflict flags
# --------------------------------------------------------------------------

def pick_targets(inst: MDAG, max_targets: int = 2) -> list[tuple]:
    """Means of partially observed variables (deterministic order), falling
    back to variable 0 when everything is always/never observed."""
    jt = inst.joint_table()
    patterns = inst.realized_patterns(jt=jt)
    partial = [i for i in range(inst.n_vars)
               if any(r[i] == 0 for r in patterns) and any(r[i] == 1 for r in patterns)]
    chosen = partial[:max_targets]
    if not chosen:
        chosen = [0]
    return [("mean", j) for j in chosen]


def conflict_flags(inst: MDAG) -> dict:
    """Population-level pattern-conflict diagnostics.

    mcar_section_violation: two realized patterns observing the same variable
    disagree on the population-marginal law of that variable (the Phase-1
    marginal-sheaf section failure; MCAR-type ignorability characterization).
    """
    jt = inst.joint_table()
    q = inst.observed_laws(jt)
    violation = False
    max_gap = 0.0
    for i in range(inst.n_vars):
        dists = []
        for r, cells in q.items():
            if r[i] != 1:
                continue
            posn = sum(1 for j in range(inst.n_vars) if r[j] == 1 and j < i)
            mg: dict[tuple, float] = {}
            for o, c in cells.items():
                mg[o[posn]] = mg.get(o[posn], 0.0) + c
            dists.append(mg)
        for d in dists[1:]:
            gap = max(abs(d.get(k, 0.0) - dists[0].get(k, 0.0))
                      for k in set(d) | set(dists[0]))
            max_gap = max(max_gap, gap)
            if gap > 1e-9:
                violation = True
    return {
        "conflict_mcar_style": bool(violation),
        "max_cross_pattern_marginal_gap": float(max_gap),
    }


# --------------------------------------------------------------------------
# mandated named classes (gate-memo carry-forward obligation)
# --------------------------------------------------------------------------

def named_structures() -> dict[str, tuple[dict[int, tuple[int, ...]], tuple]]:
    """Mechanism families the Phase-1 memo requires in the enumeration:
    mutual selection, double self-censoring, mediated MNAR, plain self
    censoring, MAR anchor, MCAR reference."""
    two_var = {0: (), 1: (0,)}
    indep2 = {0: (), 1: ()}
    three_chain = {0: (), 1: (0,), 2: (1,)}
    out = {
        "mutual_selection": (indep2, ((), (1,), (0,))),
        "double_self_censor": (two_var, ((0,), (1,))),
        "self_censor_v1": (indep2, ((0,), ())),
        "self_censor_v2_chain": (two_var, ((), (1,))),
        "mediated_mnar": (two_var, ((1,), (0, 1))),
        "mnar_on_partial_cause": (two_var, ((), (0,))),
        "mar_textbook": (two_var, ((), (0,))),
        "mcar_reference": (two_var, ((), ())),
        "three_var_mixed": (three_chain, ((), (0,), (2,))),
        "three_var_double_self": (three_chain, ((0,), (1,), (2,))),
        "collider_selection": (indep2, ((), (0, 1))),
    }
    return out

"""Gluing and obstruction instruments (WP2.3).

Discrete layer: mass-carrying tables W_r on realized patterns; the section
condition is cover-agreement (linear marginalization); global completion is
the existence of ONE full table T >= 0 whose slice onto each O(r) equals
W_r. Stalk CI constraints restrict which family members are admissible but
do not enter the gluing maps, so obstruction EXISTENCE for a poset is a
property of the linear slice system alone; constraints only gate which
families arise as observed data. Both facts are exploited here:
  - scan_poset_discrete samples mutually-consistent families and certifies
    (in)completability by LP (HiGHS), hunting genuine higher obstructions
    (pairwise-consistent, globally infeasible);
  - acyclic overlap hypergraphs are predicted completable (Graham test).

Gaussian layer: covariance-valued stalks on antichain/cyclic posets carry
assigned pairwise correlations on unit margins; global completion is PSD
feasibility of the partial correlation matrix. min-eigenvalue maximization
over free entries yields exact certificates (Phase-1 style).
"""

import itertools

import numpy as np
from scipy.optimize import linprog, minimize


# --------------------------------------------------------------------------
# discrete marginal problem
# --------------------------------------------------------------------------

def full_cells(n_vars: int) -> list[tuple]:
    return list(itertools.product((0, 1), repeat=n_vars))


def slice_marginal(T: dict[tuple, float], r: tuple[int, ...]) -> dict[tuple, float]:
    """Marginalize a FULL-variable-keyed table T onto O(r)."""
    idx = [i for i in range(len(r)) if r[i] == 1]
    out: dict[tuple, float] = {}
    for v, c in T.items():
        key = tuple(v[i] for i in idx)
        out[key] = out.get(key, 0.0) + c
    return out


def slice_marginal_dense(tab: dict[tuple, float], r: tuple[int, ...],
                         keep_vars: tuple[int, ...]) -> dict[tuple, float]:
    """Marginalize a DENSE-keyed table (keys over O(r)) onto keep_vars."""
    pos = {i: k for k, i in enumerate(j for j in range(len(r)) if r[j] == 1)}
    out: dict[tuple, float] = {}
    for o, c in tab.items():
        key = tuple(o[pos[i]] for i in keep_vars)
        out[key] = out.get(key, 0.0) + c
    return out


def marginal_problem_lp(n_vars: int, family: dict[tuple, dict[tuple, float]]) -> dict:
    """Feasibility of T on {0,1}^n with slice(T, O(r)) == family[r] for all r.

    Returns {feasible, status}. LP variables: cells of T (2^n); equality rows:
    total mass plus one row per family cell."""
    cells = full_cells(n_vars)
    nc = len(cells)
    cell_index = {v: k for k, v in enumerate(cells)}
    rows, rhs = [], []
    rows.append(np.ones(nc))
    rhs.append(1.0)
    for r, tab in family.items():
        idx = [i for i in range(n_vars) if r[i] == 1]
        for o, mass in tab.items():
            row = np.zeros(nc)
            for v in cells:
                if tuple(v[i] for i in idx) == tuple(o):
                    row[cell_index[v]] = 1.0
            rows.append(row)
            rhs.append(float(mass))
    A_eq, b_eq = np.array(rows), np.array(rhs)
    bounds = [(0.0, 1.0)] * nc
    res = linprog(np.zeros(nc), A_eq=A_eq, b_eq=b_eq, bounds=bounds,
                  method="highs")
    return {"feasible": bool(res.status == 0), "status": res.status}


def sample_family(patterns: list[tuple[int, ...]], rng: np.random.Generator,
                  concentration: float = 1.0) -> dict[tuple, dict[tuple, float]]:
    fam = {}
    for r in patterns:
        k = sum(r)
        raw = rng.gamma(concentration, 1.0, size=2 ** k)
        raw = raw / raw.sum()
        keys = list(itertools.product((0, 1), repeat=k))
        fam[r] = {kk: float(v) for kk, v in zip(keys, raw)}
    return fam


def mutually_consistent(family: dict[tuple, dict[tuple, float]],
                        tol: float = 1e-10) -> bool:
    """Agreement of every pair of family tables on their shared observed set
    (as MASS tables, including totals on the overlap)."""
    pats = list(family.keys())
    for i, a in enumerate(pats):
        for b in pats[i + 1:]:
            shared = tuple(j for j in range(len(a)) if a[j] == 1 and b[j] == 1)
            if not shared:
                continue
            ma = slice_marginal_dense(family[a], a, shared)
            mb = slice_marginal_dense(family[b], b, shared)
            if set(ma) != set(mb):
                return False
            for k in ma:
                if abs(ma[k] - mb[k]) > tol:
                    return False
    return True


def cover_consistent(poset_covers: list[tuple[tuple, tuple]],
                     family: dict[tuple, dict[tuple, float]],
                     tol: float = 1e-10) -> bool:
    for small, big in poset_covers:
        if small not in family or big not in family:
            continue
        shared = tuple(j for j in range(len(big))
                       if big[j] == 1 and small[j] == 1)
        if not shared:
            continue
        pushed = slice_marginal_dense(family[big], big, shared)
        small_dense_vars = tuple(j for j in range(len(small)) if small[j] == 1)
        target = slice_marginal_dense(family[small], small, small_dense_vars)
        for o, c in pushed.items():
            if abs(c - target.get(o, 0.0)) > tol:
                return False
    return True


def sample_pair_family(patterns: list[tuple[int, ...]],
                       rng: np.random.Generator) -> dict[tuple, dict[tuple, float]]:
    """Constructive sampler of MUTUALLY CONSISTENT families on pair-stalk
    antichains: draw singleton margins once, realize each pair table inside
    its Fréchet bounds with a uniform association parameter. Every such
    family is mutually consistent by construction."""
    n_vars = len(patterns[0])
    m = rng.dirichlet(np.ones(2), size=n_vars)
    fam: dict[tuple, dict[tuple, float]] = {}
    for r in patterns:
        idx = [i for i in range(n_vars) if r[i] == 1]
        assert len(idx) == 2, f"pair-stalk sampler got pattern {r}"
        i, j = idx
        pi, pj = m[i][1], m[j][1]
        lo = max(0.0, pi + pj - 1.0)
        hi = min(pi, pj)
        span = max(hi - lo - 2e-6, 0.0)
        t = lo + 1e-6 + rng.random() * span
        cells = {(1, 1): t, (1, 0): pi - t, (0, 1): pj - t,
                 (0, 0): 1.0 - pi - pj + t}
        assert min(cells.values()) > -1e-12
        fam[tuple(r)] = cells
    return fam


def scan_poset_discrete(patterns: list[tuple[int, ...]], n_families: int = 40,
                        seed: int = 0, sampler: str = "pair") -> dict:
    """Sample mutually-consistent families, LP-test global completability,
    count genuine higher obstructions (mutually consistent yet globally
    infeasible)."""
    rng = np.random.default_rng(seed)
    n_vars = len(patterns[0])
    n_tested = 0
    n_feasible = 0
    witness = None
    for _ in range(n_families):
        if sampler == "pair":
            fam = sample_pair_family(sorted(patterns), rng)
        else:
            fam = sample_family(sorted(patterns), rng)
            if not mutually_consistent(fam):
                continue
        res = marginal_problem_lp(n_vars, fam)
        n_tested += 1
        if res["feasible"]:
            n_feasible += 1
        elif witness is None:
            witness = {"".join(map(str, r)): {",".join(map(str, o)): float(round(float(m), 6))
                                              for o, m in tab.items()}
                       for r, tab in fam.items()}
    return {
        "patterns": [list(p) for p in sorted(patterns)],
        "sampler": sampler,
        "n_families_tested": n_tested,
        "n_globally_feasible": n_feasible,
        "n_obstructed": n_tested - n_feasible,
        "witness": witness,
    }


# --------------------------------------------------------------------------
# Gaussian (covariance-stalk) PSD-completion certificates
# --------------------------------------------------------------------------

def psd_completion_min_eig(k: int, assigned: dict[tuple[int, int], float],
                           n_starts: int = 8, seed: int = 0) -> dict:
    """Max over free entries of the minimal eigenvalue of the correlation
    matrix with prescribed entries. Positive optimum => completable (explicit
    glue returned); negative even at optimum => certified obstruction."""
    free_pairs = [(i, j) for i in range(k) for j in range(i + 1, k)
                  if (i, j) not in assigned]

    def min_eig_of(x_free):
        C = np.eye(k)
        for (i, j), rho in assigned.items():
            C[i, j] = C[j, i] = rho
        for (i, j), val in zip(free_pairs, x_free):
            C[i, j] = C[j, i] = val
        return float(np.min(np.linalg.eigvalsh(C))), C

    def obj(x):
        return -min_eig_of(x)[0]

    rng = np.random.default_rng(seed)
    best_val, best_C = -np.inf, None
    x_init = np.zeros(len(free_pairs))
    starts = [x_init]
    for _ in range(max(1, n_starts - 1)):
        starts.append(rng.uniform(-0.95, 0.95, size=len(free_pairs)))
    for x0 in starts:
        if len(free_pairs) == 0:
            val, C = min_eig_of(np.array([]))
        else:
            res = minimize(obj, x0, method="L-BFGS-B",
                           bounds=[(-0.999999, 0.999999)] * len(free_pairs),
                           options={"maxiter": 500})
            val, C = min_eig_of(res.x)
        if val > best_val:
            best_val, best_C = val, C
    return {
        "k": k,
        "assigned": {f"{i}-{j}": rho for (i, j), rho in assigned.items()},
        "optimal_min_eigenvalue": float(best_val),
        "completable": bool(best_val > 1e-9),
        "certificate_matrix": None if best_C is None else best_C.tolist(),
    }


def canonical_cycle_cases() -> list[dict]:
    """Deterministic WP2.3 witnesses/controls, expectations verified
    numerically (min-eigenvalue optima): constant and sign-alternating 4-cycles
    complete (min-eig = 1 - |c| at the symmetric optimum); obstruction needs
    incompatible path-implied correlations (mixed case), or the Phase-1
    triangle configuration."""
    cases = []

    def add(name, poset_desc, k, assigned, expect):
        res = psd_completion_min_eig(k, assigned, seed=3)
        res.update({"name": name, "poset": poset_desc,
                    "expected": expect,
                    "matches_expectation": (
                        (res["completable"] and expect == "GLUES") or
                        ((not res["completable"]) and expect == "OBSTRUCTED"))})
        cases.append(res)

    add("cycle4_const_0.9", "[[12],[23],[34],[14]]", 4,
        {(0, 1): 0.9, (1, 2): 0.9, (2, 3): 0.9, (0, 3): 0.9}, "GLUES")
    add("cycle4_const_0.5", "[[12],[23],[34],[14]]", 4,
        {(0, 1): 0.5, (1, 2): 0.5, (2, 3): 0.5, (0, 3): 0.5}, "GLUES")
    add("cycle4_alternating_0.9", "[[12],[23],[34],[14]]", 4,
        {(0, 1): 0.9, (1, 2): -0.9, (2, 3): 0.9, (0, 3): -0.9}, "GLUES")
    add("cycle4_mixed_witness", "[[12],[23],[34],[14]]", 4,
        {(0, 1): 0.95, (1, 2): 0.95, (2, 3): -0.95, (0, 3): 0.0}, "OBSTRUCTED")
    add("triangle_control_O2", "[[12],[13],[23]] with CI rho12=0", 3,
        {(0, 1): 0.0, (0, 2): 0.5, (1, 2): 0.5}, "GLUES")
    add("triangle_obstruction_O1", "[[12],[13],[23]] with CI rho12=0", 3,
        {(0, 1): 0.0, (0, 2): 0.9, (1, 2): 0.9}, "OBSTRUCTED")
    return cases

"""WP2.5.1 degeneracy null battery.

Scores null policies against the certificate's labels on the engine-undecided
rows of the frozen Phase-2 merge:

  N0  constant RECOVERABLE
  N1  fraction-observed threshold sweep (predict RECOVERABLE above tau)
  N2  pattern-overlap density threshold sweep (predict RECOVERABLE above tau)
  N3  Frechet-width sign: share-pinned assumption-free interval on the target
      mean; wide interval -> predict UNRECOVERABLE
  N4  constant UNRECOVERABLE control

The share-pinned LP matches each realized pattern's observed conditional law
scaled by its OBSERVED pattern probability P(R=r). This is the honest
Manski-style partial-identification bound given the full fingerprint. It is
deliberately NOT `lp_ground_truth.lp_range`: that Phase-1/2 relaxation omits
the share constraints and its total-mass row sums cells AND scales, which
double-counts stratum mass and squeezes every width by the phantom factor
above (stored Phase-2 `lp_width` values live on that artificial scale, hence
the uniform 0.5s). Verdicts are unaffected (width-0 maps to width-0), but all
Phase-2.5 widths are computed here on the corrected scale.

Headline output is NOT raw agreement (N0 reproduces >=99.9% of labels by
construction): it is the disagreement set S* -- the certificate-RECOVERABLE
rows with the widest corrected Frechet intervals, i.e. the boldest claims --
which becomes the priority audit sample for WP2.5.2.
"""

import itertools
import json

import numpy as np
from scipy.optimize import linprog


def frechet_bounds(n_vars: int, q: dict, pp: dict, target) -> dict:
    """Share-pinned assumption-free min/max of a mean target.

    LP variables are joint cells t[v, r] (the honest Manski-style object:
    R-strata are separate population cells, so NO cross-stratum consistency
    is imposed). Constraints: total mass 1; per realized pattern r the share
    sum_v t[v, r] = pp[r]; and the observed conditional law
    sum_{v: v_O = o} t[v, r] = pp[r] * q_r(o). The true P(v, r) is always
    feasible, so bounds bracket the truth by construction."""
    patterns = sorted(q.keys())
    cells = list(itertools.product(
        itertools.product((0, 1), repeat=n_vars), patterns))
    cindex = {c: k for k, c in enumerate(cells)}
    j = target[1]
    if target[0] != "mean":
        raise ValueError("frechet_bounds supports mean targets only")
    if not any(r[j] == 1 for r in patterns):
        return {"lo": 0.0, "hi": 1.0, "width": 1.0, "lp_status": 0,
                "degenerate": "target never observed"}

    rows = [np.ones(len(cells))]
    rhs = [1.0]
    for r in patterns:
        share_row = np.zeros(len(cells))
        for v in itertools.product((0, 1), repeat=n_vars):
            share_row[cindex[(v, r)]] = 1.0
        rows.append(share_row)
        rhs.append(float(pp.get(r, 0.0)))
        Oidx = [i for i in range(n_vars) if r[i] == 1]
        for o in itertools.product((0, 1), repeat=len(Oidx)):
            row = np.zeros(len(cells))
            for v in itertools.product((0, 1), repeat=n_vars):
                if tuple(v[i] for i in Oidx) == tuple(o):
                    row[cindex[(v, r)]] = 1.0
            rows.append(row)
            rhs.append(float(pp.get(r, 0.0)) * float(q[r].get(tuple(o), 0.0)))
    A = np.array(rows)
    b = np.array(rhs)

    c_obj = np.array([float(v[j]) for v, _ in cells])
    vals = {}
    for name, d in (("lo", 1.0), ("hi", -1.0)):
        res = linprog(d * c_obj, A_eq=A, b_eq=b,
                      bounds=[(0.0, 1.0)] * len(cells), method="highs")
        if res.status != 0:
            return {"lo": None, "hi": None, "width": None, "lp_status": res.status}
        vals[name] = float(res.fun * d)
    return {"lo": vals["lo"], "hi": vals["hi"],
            "width": vals["hi"] - vals["lo"], "lp_status": 0}


def instance_from_row(row: dict):

    vp = {int(k): tuple(v) for k, v in row["var_parents"].items()}
    structure = (vp, tuple(tuple(p) for p in row["r_parents"]))
    inst = instantiate(structure, seed=row["seed"])
    m = unpack(inst, pack(inst))
    jt = m.joint_table()
    q = m.observed_laws(jt)
    pp: dict = {}
    for (v, r), p in jt.items():
        pp[r] = pp.get(r, 0.0) + p
    return inst, q, pp


def fraction_observed(pp: dict, n_vars: int) -> float:
    tot = sum(pp.values())
    if tot <= 0:
        return 0.0
    acc = 0.0
    for r, w in pp.items():
        acc += w * sum(r)
    return acc / (tot * n_vars)


def overlap_density(patterns: list[tuple]) -> float:
    js = []
    for i, a in enumerate(patterns):
        sa = {k for k, bit in enumerate(a) if bit == 1}
        for b in patterns[i + 1:]:
            sb = {k for k, bit in enumerate(b) if bit == 1}
            union = sa | sb
            if union and sa & sb:
                js.append(len(sa & sb) / len(union))
    return float(np.mean(js)) if js else 0.0


def score_row(row: dict, tau_cfg: dict | None = None) -> dict:
    """Per-row null features + corrected Frechet interval (streaming-friendly;
    used verbatim by the Colab battery and audit notebooks)."""
    inst, q, pp = instance_from_row(row)
    fb = frechet_bounds(inst.n_vars, q, pp, tuple(row["target"]))
    return {
        "instance_id": row["instance_id"],
        "target": list(row["target"]),
        "n_vars": row["n_vars"],
        "sheaf_recoverable": row["sheaf_recoverable"],
        "frac_observed": round(fraction_observed(pp, row["n_vars"]), 6),
        "overlap_density": round(overlap_density(
            [tuple(p) for p in row["patterns"]]), 6),
        "frechet_lo": fb["lo"],
        "frechet_hi": fb["hi"],
        "frechet_width": fb["width"],
        "true_value": row.get("true_value"),
    }


def aggregate_results(scored: list[dict], cfg: dict | None = None) -> dict:
    cfg = cfg or {}
    tau_frac = cfg.get("tau_frac_observed",
                       [round(0.30 + 0.02 * k, 2) for k in range(36)])
    tau_overlap = cfg.get("tau_overlap", [round(0.05 * k, 2) for k in range(21)])
    tau_width = cfg.get("tau_width", [1e-3, 0.05, 0.10, 0.15, 0.20, 0.25,
                                      0.30, 0.35, 0.40, 0.45])
    s_cap = int(cfg.get("priority_sample_cap", 200))
    label = lambda r: r["sheaf_recoverable"] == "RECOVERABLE"  # noqa: E731

    def confusion(pred_rec):
        tp = sum(1 for p, r in zip(pred_rec, scored) if p and label(r))
        tn = sum(1 for p, r in zip(pred_rec, scored) if not p and not label(r))
        fp = sum(1 for p, r in zip(pred_rec, scored) if p and not label(r))
        fn = sum(1 for p, r in zip(pred_rec, scored) if not p and label(r))
        n = max(len(scored), 1)
        return {"TP": tp, "TN": tn, "FP": fp, "FN": fn,
                "accuracy": (tp + tn) / n}

    metrics: dict = {"n_rows": len(scored)}
    metrics["N0_constant_recoverable"] = confusion([True] * len(scored))
    metrics["N4_constant_unrecoverable"] = confusion([False] * len(scored))
    best = {"N1": None, "N2": None, "N3": None}
    sweeps = {"N1": [], "N2": [], "N3": []}
    for tau in tau_frac:
        c = confusion([r["frac_observed"] >= tau for r in scored])
        sweeps["N1"].append({"tau": tau, **c})
        if best["N1"] is None or c["accuracy"] > best["N1"]["accuracy"]:
            best["N1"] = {"tau": tau, **c}
    for tau in tau_overlap:
        c = confusion([r["overlap_density"] >= tau for r in scored])
        sweeps["N2"].append({"tau": tau, **c})
        if best["N2"] is None or c["accuracy"] > best["N2"]["accuracy"]:
            best["N2"] = {"tau": tau, **c}
    for tau in tau_width:
        c = confusion([not (r["frechet_width"] is not None
                            and r["frechet_width"] > tau) for r in scored])
        sweeps["N3"].append({"tau": tau, **c})
        if best["N3"] is None or c["accuracy"] > best["N3"]["accuracy"]:
            best["N3"] = {"tau": tau, **c}
    metrics["best_swept"] = best
    metrics["sweeps"] = sweeps

    by_n = {}
    for n in (2, 3, 4):
        sub = [r for r in scored if r["n_vars"] == n]
        ws = [r["frechet_width"] for r in sub if r["frechet_width"] is not None]
        by_n[f"n={n}"] = {
            "rows": len(sub),
            "width_min": float(np.min(ws)) if ws else None,
            "width_median": float(np.median(ws)) if ws else None,
            "width_max": float(np.max(ws)) if ws else None,
        }
    metrics["frechet_width_by_n"] = by_n

    cand = [r for r in scored if label(r) and r["frechet_width"] is not None]
    cand.sort(key=lambda r: (-r["frechet_width"], r["instance_id"],
                             json.dumps(r["target"])))
    priority = [dict(r, reason="widest_frechet_vs_certificate")
                for r in cand[:s_cap]]
    discordant = [dict(r, reason="certificate_unrecoverable_engine_undecided")
                  for r in scored if not label(r)]
    metrics["S_star_size"] = len(priority)
    metrics["S_star_rule"] = (f"top-{s_cap} certificate-RECOVERABLE rows by "
                              "corrected Frechet width (ties: id, target)")
    return {"metrics": metrics, "priority_sample": priority + discordant}


def evaluate_nulls(rows: list[dict], cfg: dict | None = None) -> dict:
    cfg = cfg or {}
    scored = [score_row(row, cfg) for row in rows]
    out = aggregate_results(scored, cfg)
    out["scored"] = scored
    return out

"""Phase-2 ground-truth decision flow (WP2.1/WP2.2 engine side).

Instruments, in precedence order (all reuse frozen Phase-1 primitives):

1. Formula oracle: every registered identification formula is attempted and
   ACCEPTED only if it reproduces the true target at machine precision on
   this instance's seeded parameters. A structurally invalid formula fails
   verification with probability 1 under random parameters, so acceptance is
   a sound positive certificate for the instance.
2. LP pinching: if the assumption-free relaxation over full tables matching
   the observed fingerprint has width ~0, the target is uniquely determined
   with NO mechanism assumptions, hence certainly recoverable under the model.
3. Model-valid witness search: two independent rounds of null-space
   root-jumping (multistart least-squares root finding on the observable
   fingerprint). A pair of distinct factorized models with identical
   fingerprints but different target values certifies unrecoverability under
   the model. NOTE (deviation from Phase 1): the SLSQP-based witness path was
   REMOVED after scipy 1.17.1's SLSQP wrapper deterministically corrupted the
   interpreter heap on certain instances (witness: structure n3_s00019_d0);
   Phase 1 itself found root-jumping the most effective witness strategy at
   these sizes.

Anything else is UNDETERMINED (sub-classified as relaxed-fragile when the LP
relaxation itself varies). Undecided rows are reported separately; primary
agreement statistics use decidable rows only.
"""

import numpy as np



def formula_oracle(inst, theta_true: np.ndarray, target) -> str | None:
    """First registered formula that verifies against the true value."""
    m = unpack(inst, theta_true)
    jt = m.joint_table()
    q = m.observed_laws(jt)
    pp = {}
    for (v, r), p in jt.items():
        pp[r] = pp.get(r, 0.0) + p
    aux = {"pattern_prob": pp}
    true_phi = target_value_phi(m, target)
    for name, fn in IDENTITY_FORMULAS.items():
        try:
            est = fn(m, q, target, aux)
        except Exception:
            continue
        if est is None or not np.isfinite(est):
            continue
        if abs(est - true_phi) <= 1e-8 * max(1.0, abs(true_phi)):
            return name
    return None


def collect_roots_early(inst, theta_ref, patterns, n_starts: int = 48,
                        max_roots: int = 12, seed: int = 0,
                        tol: float = 1e-9):
    """Distinct factorized completions of the observed fingerprint. Runs the
    FULL start budget unless max_roots distinct roots are already found (no
    duplicate-streak early stopping: converging repeatedly to one root does
    not certify uniqueness, and treating it as such produced false positives
    in piloting)."""
    from scipy.optimize import least_squares


    lo, hi = param_bounds(inst)
    free = np.where(hi - lo > 0)[0]
    base = pack(inst)
    f_ref, _ = observed_vector(unpack(inst, base), patterns)

    def expand(xf):
        th = base.copy()
        th[free] = xf
        return th

    roots = []
    if len(free) == 0:
        return [base]
    rng = np.random.default_rng(seed)
    span = hi[free] - lo[free]
    for _ in range(n_starts):
        x0f = lo[free] + 0.02 * span + rng.random(len(free)) * (0.96 * span)
        res = least_squares(
            lambda xf: observed_vector(unpack(inst, expand(xf)), patterns)[0] - f_ref,
            x0f, bounds=(lo[free], hi[free]), xtol=1e-15, ftol=1e-15, gtol=1e-15)
        if np.max(np.abs(res.fun)) >= tol:
            continue
        full = expand(res.x)
        if all(np.max(np.abs(full - u)) > 1e-6 for u in roots):
            roots.append(full.copy())
            if len(roots) >= max_roots:
                break
    return roots


def fingerprint_jacobian_rank(inst, theta: np.ndarray, patterns,
                              eps: float = 1e-6) -> tuple[int, int]:
    """Local identifiability annotation: rank of the observable-fingerprint
    Jacobian w.r.t. free parameters versus the number of free parameters.
    Descriptive evidence only; never upgrades verdicts."""

    f0, _ = observed_vector(unpack(inst, theta), patterns)
    lo, hi = param_bounds(inst)
    free = np.where(hi - lo > 0)[0]
    J = np.zeros((len(f0), len(free)))
    for k, idx in enumerate(free):
        tp = theta.copy()
        tp[idx] += eps
        tm = theta.copy()
        tm[idx] -= eps
        fp, _ = observed_vector(unpack(inst, tp), patterns)
        fm, _ = observed_vector(unpack(inst, tm), patterns)
        J[:, k] = (fp - fm) / (2 * eps)
    s = np.linalg.svd(J, compute_uv=False)
    return int(np.sum(s > 1e-8)), int(len(free))


def sheaf_fiber_verdict(inst, theta_true: np.ndarray, target,
                        n_starts: int = 48, max_roots: int = 12,
                        spread_tol: float = 1e-6, seed: int = 7) -> dict:
    """B1-side verdict: spread of the target across distinct factorized
    completions of the true observed fingerprint (fiber constancy)."""

    m_true = unpack(inst, theta_true)
    patterns = m_true.realized_patterns(jt=m_true.joint_table())
    phi_ref = target_value_phi(m_true, target)
    roots = collect_roots_early(inst, theta_true, patterns,
                                n_starts=n_starts, max_roots=max_roots,
                                seed=seed)
    phis = [target_value_phi(unpack(inst, r), target) for r in roots]
    spread = float(max(phis) - min(phis)) if phis else 0.0
    rank, n_free = fingerprint_jacobian_rank(inst, theta_true, patterns)
    return {
        "sheaf_verdict": "RECOVERABLE" if spread < spread_tol else "UNRECOVERABLE",
        "phi_spread_over_fiber": spread,
        "n_distinct_completions": len(roots),
        "phi_values_sample": [float(p) for p in phis[:max_roots]],
        "jacobian_rank": rank,
        "n_free_params": n_free,
        "n_patterns": len(patterns),
    }


def decide2(inst, theta_true: np.ndarray, target,
            jump_starts: int = 40,
            lp_pinch_tol: float = 1e-9, lp_width_tol: float = 1e-3,
            seed: int = 0) -> dict:
    """Engine-side ground truth. See module docstring for precedence."""
    out: dict = {}
    fname = formula_oracle(inst, theta_true, target)
    if fname is not None:
        out.update(gt_verdict="RECOVERABLE", gt_evidence=f"formula:{fname}")
        return out

    m_true = unpack(inst, theta_true)
    q = m_true.observed_laws()
    true_phi = target_value_phi(m_true, target)
    out["true_value"] = true_phi

    if target[0] in ("mean", "cell"):
        rng_lp = lp_range(inst, q, target)
        out["lp"] = {"width": rng_lp["width"], "lo": rng_lp["lo"], "hi": rng_lp["hi"]}
        if rng_lp["width"] <= lp_pinch_tol:
            out.update(gt_verdict="RECOVERABLE", gt_evidence="lp_pinched")
            return out

    wit = root_jump_search(inst, theta_true, target,
                           n_starts=jump_starts, seed=seed)
    if not wit["success"]:
        walk = root_jump_search(inst, theta_true, target,
                                n_starts=jump_starts, seed=seed + 101)
        if walk["delta_phi"] > wit["delta_phi"]:
            wit = walk
    out["witness"] = {k: wit[k] for k in ("delta_phi", "dist", "success")}
    if wit["success"]:
        out.update(gt_verdict="UNRECOVERABLE",
                   gt_evidence=f"model_witness(rootjump) dphi={wit['delta_phi']:.4f} "
                               f"dist={wit['dist']:.1e}")
        return out

    if out.get("lp", {}).get("width", 0.0) > lp_width_tol:
        out.update(gt_verdict="UNDETERMINED_RELAXED_FRAGILE",
                   gt_evidence="no model witness; relaxation varies")
    else:
        out.update(gt_verdict="UNDETERMINED",
                   gt_evidence="no certificate either way")
    return out

"""WP2.5.2 adversarial attackers for RECOVERABLE assertions.

Three instruments, all deliberately NON-SHARED with the certificate's own
oracle (no reuse of the certificate's fiber roots, seeds, or budgets):

  A1  deepened witness search: multi-round root-jumping at ~20x Phase-2
      start budgets plus null-space manifold walks with randomized signs,
      horizons, and fresh seeds, hunting a model pair (two factorized m-graph
      completions) that matches the observed fingerprint but differs on the
      target.
  A2  completion enumeration: fresh-seed multistart root enumeration and
      constructive manifold samplers, plus randomized-objective LP vertex
      harvests of the share-pinned completion polytope (exact rational
      vertices); reports the maximal model-valid pair divergence found.
  A3  Frechet-cell certification: classical route. For every admissible
      pair/triple of realized patterns adjacent through the target variable,
      an LP over the union table asks whether the strata laws can coexist in
      one joint and whether they pin P(V_target=1). Infeasible cells are
      recorded as classical obstructions; a globally degenerate corrected
      interval is recorded as classical certification.

A CONFIRMED false RECOVERABLE requires a MODEL-VALID witness: two completions
with max fingerprint distance < dist_tol whose targets differ by > phi_tol.
A3 can corroborate or tension, never confirm by itself. SLSQP is avoided
everywhere (scipy 1.17.1 heap-corruption incident, see engine2.py).

Every attack logs attacker identity, budget consumed, and wall time so
WP2.5.6 can price certificate-vs-search per row.
"""

import itertools
import json
import time

import numpy as np



class FastFingerprint:
    """Vectorized evaluator of the observable fingerprint and mean targets.

    Semantically identical to lp_ground_truth.observed_vector /
    target_value_phi / MDAG.realized_patterns (same sorted-pattern ordering,
    same product-order conditional keys, zero-mass conditional configs
    dropped), but evaluates P(v) and P(r|v) for all 2^n configurations at
    once. This is the attackers' own evaluation path: deliberately NOT the
    certificate's evaluator."""

    def __init__(self, inst):
        import itertools as _it

        self.n = inst.n_vars
        vs = list(_it.product((0, 1), repeat=self.n))
        v_arr = np.array(vs)
        n_v = len(vs)
        self.var_sel = []
        for i in range(self.n):
            sels = []
            for k in inst.var_cpt[i].keys():
                sel = np.ones(n_v, dtype=bool)
                for loc, pi in enumerate(inst.var_parents[i]):
                    sel &= v_arr[:, pi] == k[loc]
                sels.append(sel)
            self.var_sel.append(sels)
        self.r_sel = []
        for i in range(self.n):
            sels = []
            for k in inst.r_cpt[i].keys():
                sel = np.ones(n_v, dtype=bool)
                for loc, pi in enumerate(inst.r_parents[i]):
                    sel &= v_arr[:, pi] == k[loc]
                sels.append(sel)
            self.r_sel.append(sels)

        r_all = list(_it.product((0, 1), repeat=self.n))
        self._r_index = {r: int("".join(map(str, r)), 2) for r in r_all}
        self._bits = np.array(r_all)
        self._pattern_info = {}
        for r in r_all:
            obs = tuple(i for i in range(self.n) if r[i] == 1)
            row = self._r_index[r]
            if not obs:
                self._pattern_info[r] = {"row": row, "obs": (), "groups": []}
                continue
            cols = v_arr[:, obs]
            keys = sorted({tuple(int(x) for x in cols[k])
                           for k in range(n_v)})
            groups = []
            for key in keys:
                mask = np.ones(n_v, dtype=bool)
                for loc, pi in enumerate(obs):
                    mask &= v_arr[:, pi] == key[loc]
                groups.append((np.flatnonzero(mask)))
            self._pattern_info[r] = {"row": row, "obs": obs, "groups": groups}
        self._v_arr_j = None

    def probs(self, theta: np.ndarray):
        n_v = 2 ** self.n
        bits = self._bits
        pv = np.ones(n_v)
        idx = 0
        for i in range(self.n):
            col = np.zeros(n_v)
            for ki, sel in enumerate(self.var_sel[i]):
                col[sel] = theta[idx + ki]
            idx += len(self.var_sel[i])
            pv *= np.where(bits[:, i] == 1, col, 1.0 - col)
        qr = np.ones((n_v, n_v))
        for i in range(self.n):
            col = np.zeros(n_v)
            for ki, sel in enumerate(self.r_sel[i]):
                col[sel] = theta[idx + ki]
            idx += len(self.r_sel[i])
            qr *= np.where(bits[:, i][:, None] == 1,
                           col[None, :], 1.0 - col[None, :])
        return pv, qr

    def realized_patterns(self, theta: np.ndarray) -> list[tuple]:
        pv, qr = self.probs(theta)
        pr = qr @ pv
        out = [r for r, ri in self._r_index.items() if pr[ri] > 0]
        return sorted(out)

    def fingerprint(self, theta: np.ndarray,
                    patterns: list[tuple]) -> tuple[np.ndarray, list]:
        """(values, keys) mirroring observed_vector's key ordering."""
        pv, qr = self.probs(theta)
        pr = qr @ pv
        vals, keys = [], []
        for r in patterns:
            info = self._pattern_info[r]
            mass = float(pr[info["row"]])
            keys.append(("pat", r))
            vals.append(mass)
            if not info["obs"]:
                keys.append((r, ()))
                vals.append(1.0)
                continue
            w = qr[info["row"]] * pv
            tot = float(w.sum())
            for gi, idxs in enumerate(info["groups"]):
                agg = float(w[idxs].sum()) if tot > 0 else 0.0
                val = agg / tot if tot > 0 else 0.0
                key = tuple(int(x) for x in format(gi, f"0{len(info['obs'])}b"))
                keys.append((r, key))
                vals.append(val)
        return np.array(vals), keys

    def fingerprint_values(self, theta: np.ndarray,
                           patterns: list[tuple]) -> np.ndarray:
        return self.fingerprint(theta, patterns)[0]

    def target_value(self, theta: np.ndarray, j: int) -> float:
        pv, _ = self.probs(theta)
        return float(pv @ self._bits[:, j])


def _free_mask(inst):
    lo, hi = param_bounds_public(inst)
    free = np.where(hi - lo > 0)[0]
    return free, lo, hi


def collect_roots_fast(inst, theta_ref: np.ndarray, patterns, n_starts: int,
                       max_roots: int, seed: int, tol: float = 1e-9,
                       ls_tol: float = 1e-13) -> list[np.ndarray]:
    """engine2.collect_roots_early semantics on the FastFingerprint path:
    full start budget unless max_roots distinct roots found (no
    duplicate-streak early stopping), distinctness at 1e-6."""
    from scipy.optimize import least_squares

    ff = FastFingerprint(inst)
    f_ref = ff.fingerprint_values(theta_ref, patterns)
    base = pack(inst)
    free, lo, hi = _free_mask(inst)
    roots = []
    if len(free) == 0:
        return [base]
    rng = np.random.default_rng(seed)
    span = hi[free] - lo[free]

    def resid(xf):
        t = base.copy()
        t[free] = xf
        return ff.fingerprint_values(t, patterns) - f_ref

    for _ in range(n_starts):
        x0f = lo[free] + 0.02 * span + rng.random(len(free)) * (0.96 * span)
        res = least_squares(resid, x0f, bounds=(lo[free], hi[free]),
                            xtol=ls_tol, ftol=ls_tol, gtol=ls_tol)
        if not np.all(np.isfinite(res.x)):
            continue
        t = base.copy()
        t[free] = res.x
        if float(np.max(np.abs(ff.fingerprint_values(t, patterns)
                                    - f_ref))) >= tol:
            continue
        if all(np.max(np.abs(t - u)) > 1e-6 for u in roots):
            roots.append(t.copy())
            if len(roots) >= max_roots:
                break
    return roots


def fast_manifold_walk(inst, theta_ref: np.ndarray, target,
                       n_seeds: int = 12, steps: int = 60,
                       step_size: float = 0.02, seed: int = 0,
                       dist_tol: float = 1e-9, refresh_every: int = 5,
                       phi_tol: float = 1e-4) -> dict:
    """lp_ground_truth.manifold_walk semantics (null-space predictor +
    first-order corrector, maximizing |dphi|) on the FastFingerprint path
    with Jacobian refreshes every `refresh_every` steps."""
    ff = FastFingerprint(inst)
    patterns = ff.realized_patterns(theta_ref)
    f_ref = ff.fingerprint_values(theta_ref, patterns)
    phi_ref = ff.target_value(theta_ref, target[1])
    base = pack(inst)
    free, lo_m, hi_m = _free_mask(inst)
    if len(free) == 0:
        return {"delta_phi": 0.0, "dist": np.inf, "success": False,
                "theta_pair": None}

    def full_vec(theta_free):
        t = theta_ref.copy()
        t[free] = theta_free
        return t

    def f_of_theta(t):
        return ff.fingerprint_values(t, patterns)

    J0 = _fast_jacobian(ff, base, patterns, f_ref, free)
    U, S, Vt = np.linalg.svd(J0, full_matrices=True)
    r = int(np.sum(S > 1e-8))
    Null = Vt[r:].T if r < Vt.shape[1] else np.zeros((len(free), 0))
    Jp = np.linalg.pinv(J0)

    best = {"delta_phi": 0.0, "dist": np.inf, "success": False,
            "theta_pair": None}
    rng = np.random.default_rng(seed)
    for k in range(n_seeds):
        d = Null[:, k % Null.shape[1]] if Null.size else None
        if d is None:
            break
        x = base[free].copy()
        sign = 1.0 if rng.random() < 0.5 else -1.0
        for step in range(steps):
            x = x + sign * step_size * d / max(float(np.linalg.norm(d)), 1e-12)
            x = np.clip(x, lo_m[free], hi_m[free])
            for _ in range(3):
                fx = f_of_theta(full_vec(x))
                x = x - Jp @ (fx - f_ref)
                x = np.clip(x, lo_m[free], hi_m[free])
            fx = f_of_theta(full_vec(x))
            dist = float(np.max(np.abs(fx - f_ref)))
            if dist > 1e-7:
                break
            t = full_vec(x)
            dphi = abs(ff.target_value(t, target[1]) - phi_ref)
            if dist < dist_tol and dphi > best["delta_phi"]:
                best = {"delta_phi": float(dphi), "dist": dist,
                        "success": bool(dphi > phi_tol),
                        "theta_pair": (theta_ref.copy(), t.copy())}
            if (step + 1) % refresh_every == 0:
                Jx = _fast_jacobian(ff, t, patterns, f_ref, free)
                _, Sx, Vtx = np.linalg.svd(Jx, full_matrices=True)
                rx = int(np.sum(Sx > 1e-8))
                if rx < Vtx.shape[1]:
                    Null = np.hstack([Null, Vtx[rx:].T])
    return best


def _fast_jacobian(ff, theta, patterns, f_ref, free, eps: float = 1e-6):
    cols = []
    for idx in free:
        tp = theta.copy()
        tp[idx] += eps
        tm = theta.copy()
        tm[idx] -= eps
        fp = ff.fingerprint_values(tp, patterns)
        fm = ff.fingerprint_values(tm, patterns)
        cols.append((fp - fm) / (2 * eps))
    return np.column_stack(cols) if cols else np.zeros((len(f_ref), 0))


def deepened_witness_search(inst, theta_ref: np.ndarray, target,
                            cfg: dict, seed: int) -> dict:
    """A1: escalating root-jump rounds + randomized manifold walks on the
    attackers' own evaluation path."""
    t0 = time.perf_counter()
    rounds = int(cfg.get("a1_jump_rounds", 3))
    starts_per_round = int(cfg.get("a1_starts_per_round", 200))
    walk_seeds = int(cfg.get("a1_walk_n_seeds", 16))
    walk_steps = int(cfg.get("a1_walk_steps", 80))
    phi_tol = float(cfg.get("phi_tol", 1e-4))
    dist_tol = float(cfg.get("dist_tol", 1e-9))

    ff = FastFingerprint(inst)
    patterns = ff.realized_patterns(theta_ref)
    best = {"delta_phi": 0.0, "dist": np.inf}
    starts_used = 0
    confirmed = None
    rng = np.random.default_rng(seed)
    for rnd in range(rounds):
        round_start_best = best["delta_phi"]
        res = fast_root_jump_search(inst, theta_ref, target,
                                    n_starts=starts_per_round,
                                    seed=int(rng.integers(0, 2**31)),
                                    dist_tol=dist_tol, phi_tol=phi_tol)
        starts_used += starts_per_round
        if res["delta_phi"] > best["delta_phi"]:
            best = {"delta_phi": res["delta_phi"], "dist": res["dist"]}
        if res["success"]:
            confirmed = {"route": f"A1_rootjump_r{rnd}",
                         "theta_pair": res["theta_pair"],
                         "phi_values": res["phi_values"]}
            break
        walk = fast_manifold_walk(inst, theta_ref, target,
                                  n_seeds=walk_seeds, steps=walk_steps,
                                  step_size=float(cfg.get("a1_step_size", 0.02)),
                                  seed=int(rng.integers(0, 2**31)),
                                  dist_tol=dist_tol, phi_tol=phi_tol)
        if walk["delta_phi"] > best["delta_phi"]:
            best = {"delta_phi": walk["delta_phi"], "dist": walk["dist"]}
        if walk["success"]:
            confirmed = {"route": f"A1_manifoldwalk_r{rnd}",
                         "theta_pair": walk["theta_pair"],
                         "phi_values": None}
            if walk["theta_pair"] is not None:
                a, b = walk["theta_pair"]
                confirmed["phi_values"] = (
                    ff.target_value(a, target[1]), ff.target_value(b, target[1]))
            break
        if cfg.get("a1_adaptive_stop", True) and \
                best["delta_phi"] <= 1.1 * round_start_best:
            break
    return {
        "attacker": "A1_deepened_witness",
        "confirmed_false_recoverable": confirmed is not None,
        "best_delta_phi": float(best["delta_phi"]),
        "witness": confirmed,
        "budget_starts": starts_used,
        "wall_s": time.perf_counter() - t0,
    }


def _lp_vertex_harvest(n_vars: int, q: dict, pp: dict, target,
                       n_obj: int, seed: int) -> dict:
    """Randomized-objective harvest of exact vertices of the share-pinned
    completion polytope over joint cells t[v, r] (same object as
    battery.frechet_bounds); returns diverse rational completions and their
    target spread."""
    from scipy.optimize import linprog

    rng = np.random.default_rng(seed)
    patterns = sorted(q.keys())
    cells = list(itertools.product(
        itertools.product((0, 1), repeat=n_vars), patterns))
    cindex = {c: k for k, c in enumerate(cells)}
    rows = [np.ones(len(cells))]
    rhs = [1.0]
    for r in patterns:
        share_row = np.zeros(len(cells))
        for v in itertools.product((0, 1), repeat=n_vars):
            share_row[cindex[(v, r)]] = 1.0
        rows.append(share_row)
        rhs.append(float(pp.get(r, 0.0)))
        Oidx = [i for i in range(n_vars) if r[i] == 1]
        for o in itertools.product((0, 1), repeat=len(Oidx)):
            row = np.zeros(len(cells))
            for v in itertools.product((0, 1), repeat=n_vars):
                if tuple(v[i] for i in Oidx) == tuple(o):
                    row[cindex[(v, r)]] = 1.0
            rows.append(row)
            rhs.append(float(pp.get(r, 0.0)) * float(q[r].get(tuple(o), 0.0)))
    A = np.array(rows)
    b = np.array(rhs)

    j = target[1]
    base_c = np.array([float(v[j]) for v, _ in cells])
    phis = []
    for k in range(n_obj):
        pert = rng.uniform(-0.5, 0.5, size=len(cells))
        c = base_c + 1e-3 * pert if k else base_c
        d = 1.0 if k % 2 == 0 else -1.0
        res = linprog(d * c, A_eq=A, b_eq=b,
                      bounds=[(0.0, 1.0)] * len(cells), method="highs")
        if res.status != 0:
            continue
        phis.append(float(np.dot(base_c, res.x)))
    return {"n_vertices": len(phis),
            "phi_min": min(phis) if phis else None,
            "phi_max": max(phis) if phis else None}


def completion_enumeration(inst, theta_ref: np.ndarray, target,
                           cfg: dict, seed: int) -> dict:
    """A2: fresh-seed root enumeration + manifold continuations + LP vertex
    harvest on the FastFingerprint path. Model-valid kill requires two
    enumerated completions differing on the target within tolerances."""
    t0 = time.perf_counter()
    rng = np.random.default_rng(seed)
    root_starts = int(cfg.get("a2_root_starts", 400))
    max_roots = int(cfg.get("a2_max_roots", 32))
    walk_follows = int(cfg.get("a2_walk_follows", 6))
    walk_seeds = int(cfg.get("a2_walk_n_seeds", 12))
    phi_tol = float(cfg.get("phi_tol", 1e-4))
    dist_tol = float(cfg.get("dist_tol", 1e-9))

    ff = FastFingerprint(inst)
    patterns = ff.realized_patterns(theta_ref)
    f_ref = ff.fingerprint_values(theta_ref, patterns)
    phi_ref = ff.target_value(theta_ref, target[1])
    roots = collect_roots_fast(inst, theta_ref, patterns,
                               n_starts=root_starts, max_roots=max_roots,
                               seed=int(rng.integers(0, 2**31)),
                               tol=dist_tol)
    phis = [ff.target_value(r, target[1]) for r in roots]

    confirmed = None
    spread_model = 0.0
    if len(roots) >= 2:
        hi_i = int(np.argmax(phis))
        lo_i = int(np.argmin(phis))
        pair_dist = float(np.max(np.abs(
            ff.fingerprint_values(roots[hi_i], patterns)
            - ff.fingerprint_values(roots[lo_i], patterns))))
        spread_model = abs(phis[hi_i] - phis[lo_i])
        if pair_dist < dist_tol and spread_model > phi_tol:
            confirmed = {"route": "A2_root_pair",
                         "theta_pair": (roots[lo_i], roots[hi_i]),
                         "phi_values": (phis[lo_i], phis[hi_i])}

    walks_used = 0
    if confirmed is None and roots:
        order = np.argsort([-abs(p - phi_ref) for p in phis])
        for rk in [int(k) for k in order[:walk_follows]]:
            w = fast_manifold_walk(inst, roots[rk], target,
                                   n_seeds=walk_seeds,
                                   steps=int(cfg.get("a2_walk_steps", 60)),
                                   seed=int(rng.integers(0, 2**31)),
                                   dist_tol=dist_tol, phi_tol=phi_tol)
            walks_used += 1
            if w["success"] and w["theta_pair"] is not None:
                x2 = w["theta_pair"][1]
                d2 = float(np.max(np.abs(ff.fingerprint_values(x2, patterns)
                                         - f_ref)))
                dp2 = abs(ff.target_value(x2, target[1]) - phi_ref)
                if d2 < dist_tol and dp2 > phi_tol:
                    confirmed = {"route": "A2_manifold_follow",
                                 "theta_pair": (theta_ref.copy(), x2),
                                 "phi_values": (phi_ref,
                                                ff.target_value(x2, target[1]))}
                    break
                if dp2 > spread_model:
                    spread_model = dp2

    m_ref = unpack(inst, theta_ref)
    jt = m_ref.joint_table()
    q = m_ref.observed_laws(jt)
    pp = {}
    for (v, r), p in jt.items():
        pp[r] = pp.get(r, 0.0) + p
    verts = _lp_vertex_harvest(
        inst.n_vars, q, pp, target,
        n_obj=int(cfg.get("a2_lp_vertices", 24)),
        seed=int(rng.integers(0, 2**31)))

    return {
        "attacker": "A2_completion_enumeration",
        "confirmed_false_recoverable": confirmed is not None,
        "witness": confirmed,
        "n_roots": len(roots),
        "model_spread": float(spread_model),
        "lp_vertices": verts,
        "budget_starts": root_starts,
        "walks_run": walks_used,
        "wall_s": time.perf_counter() - t0,
    }


def frechet_cell_scan(inst, q: dict, pp: dict, target,
                      cfg: dict) -> dict:
    """A3: classical Frechet-cell certification through the target variable.

    For every admissible pair/triple of realized patterns adjacent through
    the target variable (at least one observer and one non-observer of it;
    union of observed sets bounded), solve share-pinned LPs over the union
    table: feasibility of the strata system and min/max of P(V_j=1).
    """
    from scipy.optimize import linprog

    t0 = time.perf_counter()
    j = target[1]
    n_vars = inst.n_vars
    max_union = int(cfg.get("a3_max_union_vars", 4))
    max_cells = int(cfg.get("a3_max_cells", 120))
    pin_tol = float(cfg.get("a3_pin_tol", 1e-9))
    pats = sorted(q.keys())

    cells = []
    for size in (2, 3):
        for combo in itertools.combinations(pats, size):
            has_obs = any(r[j] == 1 for r in combo)
            has_miss = any(r[j] == 0 for r in combo)
            if not (has_obs and has_miss):
                continue
            U = sorted({i for r in combo for i in range(n_vars) if r[i] == 1})
            if len(U) > max_union:
                continue
            cells.append(combo)
    n_candidate = len(cells)
    if n_candidate > max_cells:
        rng = np.random.default_rng(int(cfg.get("a3_cell_seed", 20250901)))
        picks = rng.choice(n_candidate, size=max_cells, replace=False)
        cells = [cells[int(k)] for k in sorted(picks)]

    n_feasible = n_infeasible = 0
    max_width = 0.0
    witness_cell = None
    for combo in cells:
        U = sorted({i for r in combo for i in range(n_vars) if r[i] == 1})
        ucells = list(itertools.product((0, 1), repeat=len(U)))
        rows = [np.ones(len(ucells))]
        rhs = [sum(float(pp.get(r, 0.0)) for r in combo)]
        ok = True
        for r in combo:
            Oidx = [U.index(i) for i in range(n_vars) if r[i] == 1]
            for o in itertools.product((0, 1), repeat=len(Oidx)):
                row = np.zeros(len(ucells))
                for k, uc in enumerate(ucells):
                    if tuple(uc[a] for a in Oidx) == tuple(o):
                        row[k] = 1.0
                rows.append(row)
                rhs.append(float(pp.get(r, 0.0)) * float(q[r].get(tuple(o), 0.0)))
        A = np.array(rows)
        b = np.array(rhs)
        c = np.array([float(uc[U.index(j)]) for uc in ucells])
        vals = []
        for d in (1.0, -1.0):
            res = linprog(d * c, A_eq=A, b_eq=b,
                          bounds=[(0.0, 1.0)] * len(ucells), method="highs")
            if res.status != 0:
                ok = False
                break
            vals.append(float(res.fun * d))
        if not ok:
            n_infeasible += 1
            if witness_cell is None:
                witness_cell = {"patterns": [list(r) for r in combo],
                                "type": "infeasible_strata_system"}
        else:
            n_feasible += 1
            width = vals[1] - vals[0]
            max_width = max(max_width, width)
    fb = frechet_bounds(n_vars, q, pp, target)
    return {
        "attacker": "A3_frechet_cells",
        "n_cells_tested": len(cells),
        "n_cells_candidate": n_candidate,
        "n_feasible": n_feasible,
        "n_infeasible": n_infeasible,
        "max_feasible_width": float(max_width),
        "classical_witness_cell": witness_cell,
        "global_frechet": fb,
        "classically_certified_unique": bool(
            fb["width"] is not None and fb["width"] <= pin_tol),
        "wall_s": time.perf_counter() - t0,
    }


def attack_row(row: dict, cfg: dict | None = None) -> dict:
    """Full adversarial audit of one undecided x RECOVERABLE assertion.

    `row` is a frozen Phase-2 record (structure + seed + target). Rebuilds
    the instance, runs A1 -> A2 -> A3, and returns a verdict record with
    per-attacker budgets and wall times (WP2.5.6 pricing inputs).
    """
    cfg = cfg or {}
    inst, q, pp = instance_from_row(row)
    theta = pack(inst)
    target = tuple(row["target"])

    rec = {
        "instance_id": row["instance_id"],
        "target": list(target),
        "n_vars": row["n_vars"],
        "mechanism_class": row.get("mechanism_class"),
        "poset_shape": row.get("poset_shape"),
        "certificate_sheaf": row.get("sheaf_recoverable"),
        "strata": row.get("_strata", []),
    }

    a1 = deepened_witness_search(inst, theta, target, cfg,
                                 seed=_stable_seed(row, "A1"))
    rec["A1"] = {k: v for k, v in a1.items() if k != "witness"}
    rec["A1"]["has_witness"] = a1["confirmed_false_recoverable"]
    rec["_a1_witness"] = _serialize_witness(a1)

    if a1["confirmed_false_recoverable"]:
        a2 = {"attacker": "A2_completion_enumeration", "skipped": True,
              "reason": "A1 already confirmed"}
    else:
        a2 = completion_enumeration(inst, theta, target, cfg,
                                    seed=_stable_seed(row, "A2"))
        rec["_a2_witness"] = _serialize_witness(a2)
    rec["A2"] = {k: v for k, v in a2.items()
                 if k not in ("witness", "lp_vertices")}
    rec["A2"]["has_witness"] = bool(a2.get("confirmed_false_recoverable"))
    rec["A2"]["lp_vertices"] = a2.get("lp_vertices")

    a3 = frechet_cell_scan(inst, q, pp, target, cfg)
    rec["A3"] = a3

    confirmed = a1["confirmed_false_recoverable"] or \
        a2.get("confirmed_false_recoverable", False)
    wit = None
    if a1["confirmed_false_recoverable"]:
        wit = a1["witness"]
    elif a2.get("confirmed_false_recoverable"):
        wit = a2["witness"]
    rec["verdict"] = "CONFIRMED_FALSE_RECOVERABLE" if confirmed \
        else "NO_FALSE_RECOVERABLE_FOUND"
    rec["confirming_route"] = wit["route"] if wit else None
    total_wall = sum(a["wall_s"] for a in (rec["A1"], rec["A2"], rec["A3"])
                     if isinstance(a, dict) and "wall_s" in a)
    rec["total_wall_s"] = total_wall
    return rec


def _stable_seed(row: dict, tag: str) -> int:
    import zlib
    payload = (row["instance_id"] + "|" + json.dumps(row["target"]) + "|" + tag)
    return zlib.crc32(payload.encode()) % 2**31


def fast_root_jump_search(inst, theta_ref: np.ndarray, target,
                          n_starts: int, seed: int,
                          dist_tol: float = 1e-9,
                          phi_tol: float = 1e-4) -> dict:
    """Attackers' own multistart least-squares root finder on the observable
    fingerprint (FastFingerprint evaluation path). Same acceptance semantics
    as lp_ground_truth.root_jump_search: a distinct factorized model matching
    the reference fingerprint within dist_tol whose target differs by more
    than phi_tol certifies model-unrecoverability."""
    from scipy.optimize import least_squares

    ff = FastFingerprint(inst)
    patterns = ff.realized_patterns(theta_ref)
    f_ref = ff.fingerprint_values(theta_ref, patterns)
    phi_ref = ff.target_value(theta_ref, target[1] if target[0] == "mean"
                              else target[1])
    lo, hi = param_bounds_public(inst)
    free = np.where(hi - lo > 0)[0]
    base = theta_ref.copy()
    best = {"delta_phi": 0.0, "dist": np.inf, "success": False,
            "theta_pair": None, "phi_values": None}
    if len(free) == 0:
        return best
    rng = np.random.default_rng(seed)
    span = hi[free] - lo[free]
    for _ in range(n_starts):
        x0f = lo[free] + 0.02 * span + rng.random(len(free)) * (0.96 * span)

        def resid(xf):
            t = base.copy()
            t[free] = xf
            return ff.fingerprint_values(t, patterns) - f_ref

        res = least_squares(resid, x0f, bounds=(lo[free], hi[free]),
                            xtol=1e-13, ftol=1e-13, gtol=1e-13)
        if not np.all(np.isfinite(res.x)):
            continue
        t = base.copy()
        t[free] = res.x
        dist = float(np.max(np.abs(ff.fingerprint_values(t, patterns) - f_ref)))
        if dist >= dist_tol:
            continue
        dphi = abs(ff.target_value(t, target[1]) - phi_ref)
        if dphi > best["delta_phi"]:
            best = {"delta_phi": float(dphi), "dist": dist,
                    "success": bool(dphi > phi_tol),
                    "theta_pair": (theta_ref.copy(), t.copy()),
                    "phi_values": (float(phi_ref), float(ff.target_value(t, target[1])))}
            if best["delta_phi"] > 0.5:
                break
    return best


def param_bounds_public(inst):
    return param_bounds(inst)


def _serialize_witness(result: dict):
    wit = result.get("witness")
    if not wit or not wit.get("theta_pair"):
        return None
    return {"route": wit["route"], "phi_values": list(wit["phi_values"]),
            "theta_a": [float(x) for x in wit["theta_pair"][0]],
            "theta_b": [float(x) for x in wit["theta_pair"][1]]}

"""Phase 3 pivot-gate probes (WP3.0a / WP3.0b / WP3.0c).

Everything the Colab notebook fleet embeds or imports for Phase 3 lives here;
the module is deliberately self-contained relative to its sibling modules so
that scripts/make_colab_phase3.py can concatenate it into standalone runners.

WP3.0a  natural-prevalence scan utilities: realized-pattern counting on raw
        missingness masks, Berge-cyclicity readouts (reusing the frozen
        graham_acyclic), partial-overlap flags, column-permutation negative
        controls, and a fast bootstrap for dataset-level cyclic fractions.
WP3.0b  scaling probe at n=5 (n=6 arm): uniform structure sampling beyond the
        exhaustive Phase-2 space, and a timing-split replica of the exact
        Phase-2 decision pipeline (engine round1+round2 unchanged, fiber
        certificate, share-pinned Frechet features) plus fixed-budget
        attacker runs on undecided x RECOVERABLE rows.
WP3.0c  signal-validity utilities: pin-aware instance rebuilding (cyclic
        stratum rows carry indicator pins that battery.instance_from_row
        ignores), tie-corrected rank AUCs, stratified label-permutation
        nulls, and the downstream spread-vs-naive-pooling-error correlation.

No function here upgrades verdicts: all decision logic reuses the frozen
Phase-1/2 primitives with identical seeds, budgets, and tolerances.
"""

import json
import time
import zlib
import base64
from collections import Counter

import numpy as np


# --------------------------------------------------------------------------
# pin-aware instance reconstruction (cyclic-stratum rows carry fixed_cpt)
# --------------------------------------------------------------------------

def instance_from_row_fixed(row: dict):
    """battery.instance_from_row, extended to honor fixed_cpt pins.

    Cyclic-stratum records freeze indicator mechanisms at exact 0/1 values;
    ignoring them realizes the wrong pattern family (the full simplex), so
    every Phase-3 rebuild MUST go through this constructor."""

    vp = {int(k): tuple(v) for k, v in row["var_parents"].items()}
    structure = (vp, tuple(tuple(p) for p in row["r_parents"]))
    inst = instantiate(structure, seed=row["seed"],
                       fixed_cpt=row.get("fixed_cpt") or [])
    m = unpack(inst, pack(inst))
    jt = m.joint_table()
    q = m.observed_laws(jt)
    pp: dict = {}
    for (v, r), p in jt.items():
        pp[r] = pp.get(r, 0.0) + p
    return inst, q, pp


def attack_row_fixed(row: dict, cfg: dict | None = None) -> dict:
    """attack_row replica for pinned rows: identical attacker stack (A1 -> A2
    -> A3, non-shared oracles, stable seeds) but rebuilding the instance with
    instance_from_row_fixed."""

    cfg = cfg or {}
    inst, q, pp = instance_from_row_fixed(row)
    theta = pack(inst)
    target = tuple(row["target"])

    rec = {
        "instance_id": row["instance_id"],
        "target": list(target),
        "n_vars": row["n_vars"],
        "mechanism_class": row.get("mechanism_class"),
        "poset_shape": row.get("poset_shape"),
        "certificate_sheaf": row.get("sheaf_recoverable"),
        "strata": row.get("_strata", []),
        "pinned": bool(row.get("fixed_cpt")),
    }

    a1 = deepened_witness_search(inst, theta, target, cfg,
                                 seed=_stable_seed(row, "A1"))
    rec["A1"] = {k: v for k, v in a1.items() if k != "witness"}
    rec["A1"]["has_witness"] = a1["confirmed_false_recoverable"]
    rec["_a1_witness"] = _serialize_witness(a1)

    if a1["confirmed_false_recoverable"]:
        a2 = {"attacker": "A2_completion_enumeration", "skipped": True,
              "reason": "A1 already confirmed"}
    else:
        a2 = completion_enumeration(inst, theta, target, cfg,
                                    seed=_stable_seed(row, "A2"))
        rec["_a2_witness"] = _serialize_witness(a2)
    rec["A2"] = {k: v for k, v in a2.items()
                 if k not in ("witness", "lp_vertices")}
    rec["A2"]["has_witness"] = bool(a2.get("confirmed_false_recoverable"))
    rec["A2"]["lp_vertices"] = a2.get("lp_vertices")

    a3 = frechet_cell_scan(inst, q, pp, target, cfg)
    rec["A3"] = a3

    confirmed = a1["confirmed_false_recoverable"] or \
        a2.get("confirmed_false_recoverable", False)
    wit = None
    if a1["confirmed_false_recoverable"]:
        wit = a1["witness"]
    elif a2.get("confirmed_false_recoverable"):
        wit = a2["witness"]
    rec["verdict"] = "CONFIRMED_FALSE_RECOVERABLE" if confirmed \
        else "NO_FALSE_RECOVERABLE_FOUND"
    rec["confirming_route"] = wit["route"] if wit else None
    rec["total_wall_s"] = sum(a["wall_s"] for a in (rec["A1"], rec["A2"], rec["A3"])
                              if isinstance(a, dict) and "wall_s" in a)
    return rec


# --------------------------------------------------------------------------
# WP3.0b: uniform structure sampling beyond the Phase-2 space
# --------------------------------------------------------------------------

def sample_structures(n_vars: int, count: int, seed: int,
                      prefix: str) -> list[dict]:
    """Uniformly sample structures on n_vars binary variables.

    Variable DAGs are drawn uniformly from the topologically ordered family
    (identical to enumerate_structures.var_dags); each R_i parent set is
    drawn uniformly over ALL 2^n_vars subsets (identical to the
    enumerate_structures.r_mechanisms distribution Phase 2 sampled from).
    Deterministic given seed; duplicate structures rejected."""

    rng = np.random.default_rng(seed)
    vds = var_dags(n_vars)
    jobs: list[dict] = []
    seen = set()
    guard = 0
    while len(jobs) < count and guard < 200 * count:
        guard += 1
        vd = vds[int(rng.integers(0, len(vds)))]
        rp = tuple(tuple(int(i) for i in np.flatnonzero(rng.random(n_vars) < 0.5))
                   for _ in range(n_vars))
        key = (tuple(sorted((k, tuple(v)) for k, v in vd.items())), rp)
        if key in seen:
            continue
        seen.add(key)
        jobs.append({
            "iid": f"{prefix}_j{len(jobs):04d}",
            "n_vars": n_vars,
            "structure": {
                "var_parents": {str(k): list(v) for k, v in vd.items()},
                "r_parents": [list(p) for p in rp]},
            "draw_seed": int(rng.integers(0, 2 ** 31)),
        })
    return jobs


# --------------------------------------------------------------------------
# WP3.0b: timing-split replica of the Phase-2 decision pipeline
# --------------------------------------------------------------------------

def decide2_timed(inst, theta_true: np.ndarray, target,
                  jump_starts: int = 40, round2_multiplier: int = 2,
                  lp_pinch_tol: float = 1e-9, lp_width_tol: float = 1e-3,
                  seed: int = 0) -> dict:
    """engine2.decide2 with per-instrument wall times. Verdict semantics are a
    line-for-line replica: formula oracle acceptance at 1e-8 relative
    tolerance, LP pinch at lp_pinch_tol, root-jump witness rounds (fallback
    walk kept iff strictly better), relaxed-fragile threshold at
    lp_width_tol. Round 2 reruns with jump_starts * round2_multiplier and
    seed offset exactly like the Phase-2 protocol."""

    out: dict = {}
    walls = {}
    t0 = time.perf_counter()
    fname = formula_oracle(inst, theta_true, target)
    walls["formula"] = time.perf_counter() - t0
    if fname is not None:
        out.update(gt_verdict="RECOVERABLE", gt_evidence=f"formula:{fname}")
        out["walls"] = walls
        return out

    m_true = unpack(inst, theta_true)
    q = m_true.observed_laws()
    out["true_value"] = _target_value(m_true, target)

    if target[0] in ("mean", "cell"):
        t0 = time.perf_counter()
        rng_lp = lp_range(inst, q, target)
        walls["lp"] = time.perf_counter() - t0
        out["lp"] = {"width": rng_lp["width"],
                     "lo": rng_lp["lo"], "hi": rng_lp["hi"]}
        if rng_lp["width"] <= lp_pinch_tol:
            out.update(gt_verdict="RECOVERABLE", gt_evidence="lp_pinched")
            out["walls"] = walls
            return out

    t0 = time.perf_counter()
    wit = root_jump_search(inst, theta_true, target,
                           n_starts=jump_starts, seed=seed)
    walls["witness_r1"] = time.perf_counter() - t0
    if not wit["success"]:
        walk = root_jump_search(inst, theta_true, target,
                                n_starts=jump_starts, seed=seed + 101)
        if walk["delta_phi"] > wit["delta_phi"]:
            wit = walk
            walls["witness_r1_extra"] = True
    out["witness"] = {k: wit[k] for k in ("delta_phi", "dist", "success")}
    if wit["success"]:
        out.update(gt_verdict="UNRECOVERABLE",
                   gt_evidence=f"model_witness(rootjump) dphi={wit['delta_phi']:.4f} "
                               f"dist={wit['dist']:.1e}")
        out["walls"] = walls
        return out

    t0 = time.perf_counter()
    wit2 = root_jump_search(inst, theta_true, target,
                            n_starts=jump_starts * int(round2_multiplier),
                            seed=seed + 12)
    walls["witness_r2"] = time.perf_counter() - t0
    if not wit2["success"]:
        walk = root_jump_search(inst, theta_true, target,
                                n_starts=jump_starts * int(round2_multiplier),
                                seed=seed + 113)
        if walk["delta_phi"] > wit2["delta_phi"]:
            wit2 = walk
            walls["witness_r2_extra"] = True
    if wit2["delta_phi"] > out["witness"]["delta_phi"]:
        out["witness"] = {k: wit2[k] for k in ("delta_phi", "dist", "success")}
    if wit2["success"]:
        out.update(gt_verdict="UNRECOVERABLE",
                   gt_evidence="round2:" +
                               f"model_witness(rootjump) dphi={wit2['delta_phi']:.4f} "
                               f"dist={wit2['dist']:.1e}")
        out["walls"] = walls
        return out

    if out.get("lp", {}).get("width", 0.0) > lp_width_tol:
        out.update(gt_verdict="UNDETERMINED_RELAXED_FRAGILE",
                   gt_evidence="round2:no model witness; relaxation varies")
    else:
        out.update(gt_verdict="UNDETERMINED",
                   gt_evidence="round2:no certificate either way")
    out["walls"] = walls
    return out


def _target_value(m, target) -> float:
    return target_value_phi(m, target)


def run_scaling_job(job: dict, cfg: dict) -> list[dict]:
    """Full WP3.0b pipeline for one structure: engine round1(+round2),
    fiber certificate, structural annotations, share-pinned Frechet features,
    and (when job['do_attack']) a fixed-budget attacker run on undecided x
    RECOVERABLE rows. Row schema extends the Phase-2 merge schema with wall
    splits and feature fields; `job['fixed_cpt']` is honored when present."""

    budgets = cfg["budgets"]
    t0 = time.perf_counter()
    vp = {int(k): tuple(v) for k, v in job["structure"]["var_parents"].items()}
    structure = (vp, tuple(tuple(p) for p in job["structure"]["r_parents"]))
    inst = instantiate(structure, seed=job["draw_seed"],
                       fixed_cpt=job.get("fixed_cpt") or [])
    info = classify(inst)
    jt = inst.joint_table()
    patterns = inst.realized_patterns(jt=jt)
    q = inst.observed_laws(jt)
    pp: dict = {}
    for (v, r), p in jt.items():
        pp[r] = pp.get(r, 0.0) + p
    fam_w = {r: {o: c * pp[r] for o, c in cells.items()} for r, cells in q.items()}
    completability = marginal_problem_lp(inst.n_vars, fam_w)["feasible"]
    sets = [frozenset(i for i in range(inst.n_vars) if r[i] == 1) for r in patterns]
    shape = poset_shape(patterns)
    conflicts = conflict_flags(inst)
    cis = discover_slice_cis(inst, n_draws=int(budgets.get("ci_discovery_draws", 16)))
    theta_true = pack(inst)
    wall_struct = time.perf_counter() - t0

    records = []
    for tgt in pick_targets(inst):
        eng = decide2_timed(
            inst, theta_true, tgt,
            jump_starts=int(budgets["jump_starts"]),
            round2_multiplier=int(budgets.get("round2_multiplier", 2)),
            lp_pinch_tol=float(budgets.get("lp_pinch_tol", 1e-9)),
            lp_width_tol=float(budgets.get("lp_width_tol", 1e-3)),
            seed=11)
        walls = eng.pop("walls")
        undecided = eng["gt_verdict"].startswith("UNDETERMINED")

        t0 = time.perf_counter()
        fib = sheaf_fiber_verdict(inst, theta_true, tgt,
                                  n_starts=int(budgets["fiber_starts"]),
                                  max_roots=int(budgets.get("max_roots", 12)),
                                  seed=13)
        wall_fiber = time.perf_counter() - t0

        t0 = time.perf_counter()
        fb = frechet_bounds(inst.n_vars, q, pp, tgt)
        wall_frechet = time.perf_counter() - t0

        do_attack = bool(job.get("do_attack")) and undecided \
            and fib["sheaf_verdict"] == "RECOVERABLE"
        attack_rec = None
        if do_attack:
            rowlike = {
                "instance_id": job["iid"],
                "target": list(tgt),
                "n_vars": inst.n_vars,
                "var_parents": {str(k): list(v) for k, v in vp.items()},
                "r_parents": [list(p) for p in structure[1]],
                "seed": job["draw_seed"],
                "fixed_cpt": job.get("fixed_cpt"),
                "mechanism_class": info["mechanism_class"],
                "poset_shape": shape,
                "sheaf_recoverable": fib["sheaf_verdict"],
            }
            try:
                t0 = time.perf_counter()
                attack_rec = attack_row_fixed(rowlike, cfg.get("attack"))
                attack_rec["wall_dispatch_s"] = time.perf_counter() - t0
            except Exception as e:  # never lose the engine row to an attack crash
                attack_rec = {"status": "error", "error": f"{type(e).__name__}: {e}"}

        records.append({
            "instance_id": job["iid"],
            "tag": job.get("tag", f"n{inst.n_vars}"),
            "seed": job["draw_seed"],
            "template": job.get("template"),
            "fixed_cpt": job.get("fixed_cpt"),
            "n_vars": inst.n_vars,
            "var_parents": {str(k): list(v) for k, v in vp.items()},
            "r_parents": [list(p) for p in structure[1]],
            "mechanism_class": info["mechanism_class"],
            "has_self_edge": info["has_self_edge"],
            "poset_shape": shape,
            "graham_acyclic": bool(graham_acyclic(sets)),
            "n_realized_patterns": len(patterns),
            "patterns": [list(p) for p in patterns],
            "always_observed": list(info["always_observed"]),
            "never_observed": list(info["never_observed"]),
            "target": list(tgt),
            "true_value": eng.get("true_value"),
            "gt_recoverable": eng["gt_verdict"],
            "gt_evidence": eng["gt_evidence"],
            "lp_width": eng.get("lp", {}).get("width"),
            "witness_delta_phi": eng.get("witness", {}).get("delta_phi"),
            "sheaf_recoverable": fib["sheaf_verdict"],
            "phi_spread_over_fiber": fib["phi_spread_over_fiber"],
            "n_distinct_completions": fib["n_distinct_completions"],
            "jacobian_rank": fib["jacobian_rank"],
            "n_free_params": fib["n_free_params"],
            "jacobian_rank_deficiency": int(fib["n_free_params"] - fib["jacobian_rank"]),
            "jacobian_full_rank": bool(fib["jacobian_rank"] == fib["n_free_params"]),
            "observed_family_completable": bool(completability),
            "conflict_mcar_style": conflicts["conflict_mcar_style"],
            "max_cross_pattern_marginal_gap": conflicts["max_cross_pattern_marginal_gap"],
            "n_slice_ci_constraints": int(sum(len(v) for v in cis.values())),
            "frechet_lo": fb["lo"],
            "frechet_hi": fb["hi"],
            "frechet_width": fb["width"],
            "frac_observed": round(fraction_observed(pp, inst.n_vars), 6),
            "overlap_density": round(overlap_density([tuple(p) for p in patterns]), 6),
            "attack_requested": bool(job.get("do_attack")) and undecided
            and fib["sheaf_verdict"] == "RECOVERABLE",
            "attack": attack_rec,
            "wall_struct_s": round(wall_struct, 3),
            "wall_formula_s": round(walls.get("formula", 0.0), 3),
            "wall_lp_s": round(walls.get("lp", 0.0), 3),
            "wall_engine_r1_s": round(walls.get("witness_r1", 0.0), 3),
            "wall_engine_r2_s": round(walls.get("witness_r2", 0.0), 3),
            "wall_fiber_s": round(wall_fiber, 3),
            "wall_features_s": round(wall_frechet, 3),
            "wall_attack_s": round(attack_rec.get("total_wall_s", 0.0), 3)
            if isinstance(attack_rec, dict) else 0.0,
        })
    return records


# --------------------------------------------------------------------------
# WP3.0a: prevalence scan utilities
# --------------------------------------------------------------------------

def realized_pattern_counts(obs: np.ndarray) -> dict[tuple, int]:
    """Counts of realized observed-set patterns. `obs` is a boolean matrix
    (n_rows, k), True = observed; pattern tuples follow the engine convention
    (1 = observed)."""
    codes = _pattern_codes(obs)
    counts = np.bincount(codes, minlength=1 << obs.shape[1])
    out = {}
    for code in range(1 << obs.shape[1]):
        if counts[code]:
            pat = tuple((code >> i) & 1 for i in range(obs.shape[1]))
            out[pat] = int(counts[code])
    return out


def _pattern_codes(obs: np.ndarray) -> np.ndarray:
    weights = 1 << np.arange(obs.shape[1])
    return obs.astype(np.int64) @ weights


def scan_subsets(obs: np.ndarray, col_names: list[str],
                 subsets: list[tuple[int, ...]], min_patterns: int = 4,
                 min_support: int = 1) -> list[dict]:
    """Per-subset prevalence records for WP3.0a. Cyclicity is Berge-cyclicity
    of the observed-set hypergraph restricted to patterns whose support is at
    least min_support (main analysis: min_support=1, i.e., any realized
    pattern; robustness: min_support>1)."""

    records = []
    for sub in subsets:
        sub_obs = obs[:, list(sub)]
        counts = realized_pattern_counts(sub_obs)
        kept = {p: c for p, c in counts.items() if c >= min_support}
        n_patterns = len(kept)
        rec = {
            "cols": [col_names[i] for i in sub],
            "size": len(sub),
            "n_realized_patterns": n_patterns,
            "eligible": n_patterns >= min_patterns,
        }
        if rec["eligible"]:
            sets = [frozenset(i for i, bit in enumerate(p) if bit == 1)
                    for p in sorted(kept)]
            acyclic = graham_acyclic(sets)
            nested = _nested_only(sorted(kept))
            rec.update({
                "graham_acyclic": bool(acyclic),
                "cyclic": not acyclic,
                "nested_only": bool(nested),
                "partial_overlap": bool(not nested),
                "max_support_gap": int(max(kept.values()) - min(kept.values())),
            })
        records.append(rec)
    return records


def _nested_only(patterns: list[tuple]) -> bool:
    sets = [frozenset(i for i, bit in enumerate(p) if bit == 1) for p in patterns]
    return all(a <= b or b <= a for a, b in
               [(x, y) for i, x in enumerate(sets) for y in sets[i + 1:]])


def column_permutation_control(obs: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Negative control: independently permute each column's missingness mask,
    destroying co-missingness dependence while preserving marginals."""
    out = np.empty_like(obs)
    for j in range(obs.shape[1]):
        out[:, j] = obs[rng.permutation(obs.shape[0]), j]
    return out


def cyclic_fraction_bootstrap(obs: np.ndarray, subsets: list[tuple[int, ...]],
                              B: int, min_patterns: int, min_support: int,
                              seed: int) -> dict:
    """Dataset-level bootstrap of the cyclic fraction over pre-selected
    eligible subsets (eligibility frozen at the full sample; realized patterns
    recount per resample). Returns fraction quantiles and per-subset stability."""

    rng = np.random.default_rng(seed)
    n = obs.shape[0]
    fracs = []
    stable = np.zeros(len(subsets))
    for _ in range(B):
        idx = rng.integers(0, n, size=n)
        obs_b = obs[idx]
        cyc = 0
        elig = 0
        for si, sub in enumerate(subsets):
            counts = realized_pattern_counts(obs_b[:, list(sub)])
            kept = {p: c for p, c in counts.items() if c >= min_support}
            if len(kept) < min_patterns:
                continue
            elig += 1
            sets = [frozenset(i for i, bit in enumerate(p) if bit == 1)
                    for p in sorted(kept)]
            if not graham_acyclic(sets):
                cyc += 1
                stable[si] += 1
        fracs.append(cyc / max(elig, 1))
    fracs = np.array(fracs)
    return {
        "B": B,
        "fraction_q05": float(np.quantile(fracs, 0.05)),
        "fraction_median": float(np.median(fracs)),
        "fraction_q95": float(np.quantile(fracs, 0.95)),
        "per_subset_cyclic_stability": [round(float(s / B), 4) for s in stable],
    }


# --------------------------------------------------------------------------
# WP3.0c: signal-validity utilities
# --------------------------------------------------------------------------

def naive_pooling_mean(m, j: int) -> float:
    """MCAR-plugin estimate of E[V_j]: probability-weighted average of the
    pattern-conditional means over the patterns that observe j (weights are
    the population pattern probabilities, normalized among observers)."""
    jt = m.joint_table()
    q = m.observed_laws(jt)
    pp: dict = {}
    for (v, r), p in jt.items():
        pp[r] = pp.get(r, 0.0) + p
    num = den = 0.0
    for r, cells in q.items():
        if r[j] != 1:
            continue
        w = pp.get(r, 0.0)
        pos = sum(1 for kk in range(len(r)) if r[kk] == 1 and kk < j)
        mg: dict[int, float] = {}
        for o, c in cells.items():
            mg[o[pos]] = mg.get(o[pos], 0.0) + c
        num += w * sum(k * c for k, c in mg.items())
        den += w * sum(mg.values())
    return num / den if den > 0 else float("nan")


def spread_naive_table(rows: list[dict]) -> list[dict]:
    """Downstream add-on inputs: for each rebuildable row, (fiber spread,
    corrected Frechet width, cross-pattern gap) vs absolute naive-pooling
    error against the true estimand."""


    out = []
    for row in rows:
        try:
            inst, q, pp = instance_from_row_fixed(row)
            m = unpack_model(inst)
            j = int(row["target"][1])
            phi_true = target_value_phi(m, tuple(row["target"]))
            est = naive_pooling_mean(m, j)
            fwidth = row.get("frechet_width")
            if fwidth is None:
                try:
                    fb = frechet_bounds(inst.n_vars, q, pp, tuple(row["target"]))
                    fwidth = fb["width"]
                except Exception:
                    fwidth = None
            out.append({
                "instance_id": row.get("instance_id"),
                "source": row.get("source_tag", "unspecified"),
                "spread": float(row.get("phi_spread_over_fiber", 0.0) or 0.0),
                "naive_abs_err": abs(est - phi_true),
                "frechet_width": fwidth,
                "max_gap": row.get("max_cross_pattern_marginal_gap"),
            })
        except Exception:
            continue
    return out


def unpack_model(inst):
    return unpack(inst, pack(inst))


def rank_auc(scores, labels) -> float | None:
    """Tie-corrected Mann-Whitney AUC; None when only one class present."""
    s = np.asarray(scores, dtype=float)
    y = np.asarray(labels, dtype=int)
    pos = s[y == 1]
    neg = s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return None
    order = np.argsort(s, kind="mergesort")
    ranks = np.empty(len(s), dtype=float)
    sp = s[order]
    i = 0
    while i < len(sp):
        j = i
        while j + 1 < len(sp) and sp[j + 1] == sp[i]:
            j += 1
        ranks[order[i:j + 1]] = 0.5 * (i + j) + 1.0
        i = j + 1
    r_pos = ranks[y == 1].sum()
    return float((r_pos - len(pos) * (len(pos) + 1) / 2.0)
                 / (len(pos) * len(neg)))


def permutation_auc_p(scores, labels, strata, B: int,
                      seed: int) -> dict:
    """Stratified label-permutation null for the AUC: labels are shuffled
    within strata (e.g., n_vars x mechanism_class buckets), preserving class
    balance per bucket. One-sided p = P(AUC_perm >= AUC_obs) with add-one
    smoothing."""
    rng = np.random.default_rng(seed)
    y = np.asarray(labels, dtype=int)
    strata = np.asarray(strata)
    auc_obs = rank_auc(scores, y)
    if auc_obs is None:
        return {"auc": None, "p_value": None, "B": B,
                "reason": "single-class labels"}
    ge = 0
    null_aucs = []
    for _ in range(B):
        yp = np.empty_like(y)
        for st in np.unique(strata):
            mask = strata == st
            vals = y[mask]
            perm = rng.permutation(vals)
            yp[mask] = perm
        a = rank_auc(scores, yp)
        if a is not None:
            null_aucs.append(a)
            if a >= auc_obs:
                ge += 1
    return {
        "auc": auc_obs,
        "p_value": (1 + ge) / (B + 1),
        "B": B,
        "null_mean": float(np.mean(null_aucs)) if null_aucs else None,
        "null_sd": float(np.std(null_aucs)) if null_aucs else None,
    }


def permutation_corr_p(x, y, B: int, seed: int, method: str = "spearman") -> dict:
    """One-sided permutation p for |correlation| exceeding the observed value."""
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]

    def corr(a, b):
        if method == "spearman":
            a = np.argsort(np.argsort(a)).astype(float)
            b = np.argsort(np.argsort(b)).astype(float)
        if a.std() == 0 or b.std() == 0:
            return 0.0
        return float(np.corrcoef(a, b)[0, 1])

    if len(x) < 3 or not np.isfinite(x).all() or not np.isfinite(y).all():
        return {"rho": None, "p_two_sided": None, "B": B, "method": method,
                "n": int(len(x)), "reason": "insufficient finite pairs"}
    rho = corr(x, y)
    ge = 0
    for _ in range(B):
        if abs(corr(x, rng.permutation(y))) >= abs(rho):
            ge += 1
    return {"rho": rho, "p_two_sided": (1 + ge) / (B + 1),
            "B": B, "method": method, "n": int(len(x))}


# --------------------------------------------------------------------------
# compact payload codec shared by the embedded-notebook fleet
# --------------------------------------------------------------------------

ENGINE_ROW_FIELDS = [
    "instance_id", "tag", "seed", "template", "fixed_cpt", "n_vars",
    "var_parents", "r_parents", "mechanism_class", "has_self_edge",
    "poset_shape", "graham_acyclic", "n_realized_patterns", "patterns",
    "always_observed", "never_observed", "target", "true_value",
    "gt_recoverable", "gt_evidence", "sheaf_recoverable",
    "phi_spread_over_fiber", "n_distinct_completions", "jacobian_rank",
    "n_free_params", "conflict_mcar_style", "max_cross_pattern_marginal_gap",
]


def compact_engine_row(row: dict) -> dict:
    return {k: row[k] for k in ENGINE_ROW_FIELDS if k in row}


def compress_payload(obj, level: int = 9) -> str:
    blob = json.dumps(obj, separators=(",", ":")).encode()
    return base64.b64encode(zlib.compress(blob, level)).decode()


def decompress_payload(b64: str):
    return json.loads(zlib.decompress(base64.b64decode(b64)))


In [ ]:
PAYLOADS = {}
PAYLOADS['CYC_ROWS'] = (
    '...'
)
PAYLOADS['AUDIT_VERDICTS'] = (
    '...'
    'WyjqyfdxcaenXwX3VXhMDVf/f/7P/wOaDE87'
)
PAYLOADS['FAMILY_ROWS'] = (
    'eNrd3M1rXFUYBvD/5a5n8X6c9yu7akcp1ApBRQghxHYsARtlktZF6f/uueLCvmcjugiPWeQyyQTmYXLP7zl3Zt6rj9u707uf'
    'TuebuzfbBR+2N+fb328eTqd5S0giU+iwPTye379+fH8+bRcftw+355vfbs+n+8eH/SZtF1fXh43ngeZR9uOBrz8dtr/d7eqK'
    'rw/z5/Lnd57H/Q6Pt+e3p8f52/kYbu+3w/73p/u3d/enmw+n85u71/N32/evLo9ffvvD8fLZFy+P22G7e7j59Xw37/TXo/z5'
    '9peH06fDZzmk50gPLbwc2nPUcEu8HKPlUBKLwMthPQfTKMfL4UuOHGx4OaLnEFcdeDmy59Ahpng5qucYwiF4OZh6ECMqBgzC'
    'S5AkBhSdO+nqVgJIOnfTNTQHoOm8oJ4cDog6L6pneQKqzgvrFU6ArHN3fZCZPLnrz4/fHS+/efHq+Pzm8vjy2Y/z+NXls69f'
    '/PNc3fnBOgag89yhn86rA0IvtAQpSUDopUM/NIQQt+4d+jGMGRB66dAP00kLYJAO/XAqC/jVWGzJlRmA7kt3f8weVoDuy+J+'
    'jmDA/bws0Nd+ZQIwSId+7oNnFQO8kkpLkP1/CzBIh97YtQh+Ndbu/iz8SojX7Lv7piIC6L52920QD8ANvtoSJMkBodcOvZlV'
    'AkKvHXrb1zBA6LVDb8EpgNDrAn1UDEDoP3deyzmoiKF8nI90eQV16S9zq+KA/WUs/aXMEvBCxeiFxUlHARaW0Z13LlVA54cu'
    'QXgw4tslemFxCTHAwjJ6YXE1DsDCMnph8X1zAlhYRnfeLUsBnbcOorunAYJoHUSPEYG/obfFxxRPQB9tYaWm9YCs2AL93DwK'
    'IvS5LGJMrPinTHc/pjMD0H3rXMZ+CQyQS+sFJnioAxYY66dMKAkBXqjw7n4Mp4Hofi3PSLIANjHv0IdzBiD0rkuQigKE3jv0'
    'Mb8YEHpfQExzRXzDfS/5YVoOeNXLu+y5vyYEKLvnEqS0EEHsjiSHMqIjS9cqNUP8VEqvKCkmClhRoq9aqcoGuGpFryg5mByw'
    'okSXfXpYBCh76PKMZCVg14q+amVIDMRVq3et9JEC2LVikT3TElD2WLrW/rYVxI83dhCLxhBAELODWCw6EEFcSmO5EWBpzC57'
    'CYkByp66BJlFHhDE7BWl1KkAK0p2EGsMYkAQs1eUMi4FrCjZZS+PCEDZM5ZnpNIAZa9F9lRnQNlrkb3YFFD2WkCsuW4hgtgr'
    'SsXc6wJWlGogMrFJAoJY1oOICgGCWN6DKLMAglixBNm3VoBBsgcZXo44Aad6EBuZkCNwqCdxSUKcgUPckwQF4lg7JlmSpCMO'
    'tivtQSjUE3/CBy3Yz/aFOOqOqWu/c4847I6pc887+IhzcWg5bdLN/g+nTSxPUQriJD+m3mRYnBFn+TH1KsM6CHKaXx/nt38s'
    'qDDH+fGSZAZBrDJ9Dh6zW0DOweuD8JhnlYEchMcL+ckGOQmPF/KzDHIUHi8y7p/LRpSxT7/jfTQG5PS7Pox0njiRTz+M9L/X'
    'sD7Xj4VVnn6u378Idv0H/CbETg=='
)
CYC_ROWS = decompress_payload(PAYLOADS['CYC_ROWS'])
AUDIT_VERDICTS = decompress_payload(PAYLOADS['AUDIT_VERDICTS'])
FAMILY_ROWS = decompress_payload(PAYLOADS['FAMILY_ROWS'])
print('payloads:', len(CYC_ROWS), 'cyclic rows,',
      len(AUDIT_VERDICTS), 'audit verdicts,', len(FAMILY_ROWS),
      'family rows')


In [ ]:
CYCATTACK_CFG = json.loads(r'''{"description": "WP2.5.2 adversarial audit budgets. Phase 2's recorded engine costs (n=3 undecided rows: median 72 s, p90 303 s, max 677 s for round1+round2+fiber) make a literal x20 multiplier on every audited row infeasible inside the plan's own 30-60 CPU-hour envelope. Resolution per Section 11: A1 escalates ADAPTIVELY (rounds stop early on success or on no-improvement), so the x20 language is a per-row CEILING, not a flat spend; effective multiplier is data-dependent and is repriced by WP2.5.6 from the verdict logs. Every notebook self-pilots its first job and prints an extrapolation before continuing.", "sampling": {"n2_census": true, "n4_census": true, "n3_srs_N": 400, "srs_seed": 20250901, "include_S_star": true, "S_star_recomputed_in_notebook": true}, "full": {"a1_jump_rounds": 4, "a1_starts_per_round": 200, "a1_walk_n_seeds": 8, "a1_walk_steps": 60, "a1_step_size": 0.02, "a2_root_starts": 150, "a2_max_roots": 16, "a2_walk_follows": 4, "a2_walk_n_seeds": 8, "a2_walk_steps": 50, "a2_lp_vertices": 16, "a3_max_union_vars": 4, "a3_max_cells": 120, "a3_pin_tol": 1e-09, "phi_tol": 0.0001, "dist_tol": 1e-09}, "pilot": {"a1_jump_rounds": 1, "a1_starts_per_round": 20, "a1_walk_n_seeds": 2, "a1_walk_steps": 20, "a1_step_size": 0.02, "a2_root_starts": 24, "a2_max_roots": 8, "a2_walk_follows": 1, "a2_walk_n_seeds": 3, "a2_walk_steps": 20, "a2_lp_vertices": 6, "a3_max_union_vars": 4, "a3_max_cells": 40, "a3_pin_tol": 1e-09, "phi_tol": 0.0001, "dist_tol": 1e-09}, "adaptive_escalation": {"early_stop_on_success": true, "no_improvement_stop": "stop after a round whose best delta_phi improved <10% relative over the previous round, unless a witness was found", "note": "implemented in attackers.deepened_witness_search"}, "kill_rule": "any CONFIRMED false RECOVERABLE (model-valid pair: dist < dist_tol AND dphi > phi_tol on the attackers' independent FastFingerprint path) -> C1 KILL (rule D2)", "outputs": ["results/phase25/audit_sample.jsonl", "results/phase25/audit_verdicts.jsonl"], "a1_jump_rounds": 4, "a1_starts_per_round": 200, "a1_walk_n_seeds": 8, "a1_walk_steps": 60, "a1_step_size": 0.02, "a2_root_starts": 150, "a2_max_roots": 16, "a2_walk_follows": 4, "a2_walk_n_seeds": 8, "a2_walk_steps": 50, "a2_lp_vertices": 16, "a3_max_union_vars": 4, "a3_max_cells": 120, "a3_pin_tol": 1e-09, "phi_tol": 0.0001, "dist_tol": 1e-09}''')


In [ ]:
SHARD_IDX = 0
N_SHARDS = 4
OUT_DIR = pathlib.Path('/content/results/phase3')
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def _san(o):
    import numpy as _np
    if isinstance(o, (_np.floating,)):
        return float(o)
    if isinstance(o, (_np.integer,)):
        return int(o)
    if isinstance(o, (_np.bool_,)):
        return bool(o)
    if isinstance(o, _np.ndarray):
        return o.tolist()
    raise TypeError(str(type(o)))


def dump_line(obj):
    return json.dumps(obj, default=_san)


def load_done(path, key_fn):
    done = set()
    if path.exists():
        for line in path.read_text().splitlines():
            try:
                done.add(key_fn(json.loads(line)))
            except Exception:
                pass
    return done


def pooled_map_deadline(worker_fn, items, n_workers=2, stall_timeout_s=5400,
                        seconds_budget=None, meta=None):
    """Yield worker_fn(item) for all items on a fork-context pool with about
    2*n_workers futures in flight. Stops dispatching once seconds_budget is
    exhausted (running jobs drain; queued ones are cancelled and reported in
    meta['not_run']); falls back to sequential execution on pool failure.
    Job dicts may carry EITHER 'iid' or 'instance_id' as their key."""
    from concurrent.futures import ProcessPoolExecutor, FIRST_COMPLETED, wait

    items = list(items)
    if meta is None:
        meta = {}
    meta['not_run'] = []
    meta['timed_out'] = False
    meta['completed'] = 0
    meta['done_keys'] = set()
    t_start = time.time()

    def left():
        return None if seconds_budget is None else seconds_budget - (time.time() - t_start)

    def jkey(it):
        if isinstance(it, dict):
            return it.get('iid') or it.get('instance_id') or id(it)
        return str(it)

    if len(items) <= 1 or n_workers <= 1:
        for it in items:
            if left() is not None and left() <= 0:
                meta['timed_out'] = True
                meta['not_run'] = [jkey(x) for x in items[items.index(it):]]
                return
            r = worker_fn(it)
            meta['completed'] += 1
            meta['done_keys'].add(jkey(it))
            yield r
        return

    ex = ProcessPoolExecutor(max_workers=n_workers,
                             mp_context=mp.get_context('fork'))
    fut_item = {}
    nxt = 0
    try:
        while True:
            while nxt < len(items) and len(fut_item) < 2 * n_workers:
                if left() is not None and left() <= 0:
                    meta['timed_out'] = True
                    break
                f = ex.submit(worker_fn, items[nxt])
                fut_item[f] = items[nxt]
                nxt += 1
            if meta['timed_out']:
                meta['not_run'].extend(jkey(x) for x in items[nxt:])
                nxt = len(items)
            if fut_item:
                done_set, _ = wait(set(fut_item), timeout=stall_timeout_s,
                                   return_when=FIRST_COMPLETED)
                if not done_set:
                    raise RuntimeError(
                        f'pool stalled {stall_timeout_s}s with '
                        f'{len(fut_item)} futures pending')
                for f in done_set:
                    it = fut_item.pop(f)
                    meta['completed'] += 1
                    meta['done_keys'].add(jkey(it))
                    yield f.result()
            if nxt >= len(items) and not fut_item:
                break
            if meta['timed_out']:
                still = {}
                for f, it in fut_item.items():
                    if not f.cancel():
                        still[f] = it
                    else:
                        meta['not_run'].append(jkey(it))
                fut_item = still
        ex.shutdown(wait=False, cancel_futures=True)
    except Exception as e:
        print(f'(pool yielded {meta["completed"]}/{len(items)} then '
              f'{type(e).__name__}; finishing remainder sequentially)',
              flush=True)
        for proc in (getattr(ex, '_processes', None) or {}).values():
            try:
                proc.kill()
            except Exception:
                pass
        ex.shutdown(wait=False, cancel_futures=True)
        for it in items:
            if jkey(it) in meta['done_keys']:
                continue
            r = worker_fn(it)
            meta['completed'] += 1
            meta['done_keys'].add(jkey(it))
            yield r


In [ ]:
T_START = time.time()
VERDICT_PATH = OUT_DIR / f"cycattack_verdicts_shard{SHARD_IDX:02d}.jsonl"
SAMPLE_PATH = OUT_DIR / f"cycattack_sample_shard{SHARD_IDX:02d}.json"

und_rec = [r for r in CYC_ROWS
           if r.get("gt_recoverable", "").startswith("UNDETERMINED")
           and r.get("sheaf_recoverable") == "RECOVERABLE"]
frame = {}
for r in und_rec:
    key = r["instance_id"] + "|" + json.dumps(r["target"])
    frame[key] = r
keys_all = sorted(frame.keys())
jobs_all = [frame[k] for k in keys_all if k not in
            load_done(VERDICT_PATH,
                      lambda r: r["instance_id"] + "|" + json.dumps(r["target"]))]
jobs = jobs_all[SHARD_IDX::N_SHARDS]
with open(SAMPLE_PATH, "w") as f:
    json.dump({"n_undecided_rec_total": len(keys_all),
               "shard_jobs_pending": len(jobs),
               "shard": SHARD_IDX, "n_shards": N_SHARDS}, f)
print(f"cyclic undecided x REC rows: {len(keys_all)}; shard {SHARD_IDX} "
      f"pending: {len(jobs)}", flush=True)


def attack_worker(row):
    return attack_row_fixed(row, CYCATTACK_CFG)


kills = 0
meta = {}
t0 = time.time()
if jobs:
    tp0 = time.time()
    pilot = attack_worker(dict(jobs[0]))
    per = time.time() - tp0
    print(f"self-pilot: {per:.0f}s/row -> projection ~"
          f"{per * len(jobs) / 2 / 3600:.1f} h on 2 workers; continuing",
          flush=True)
    with open(VERDICT_PATH, "a") as f:
        f.write(dump_line(pilot) + "\n")
    kills += int(pilot.get("verdict") == "CONFIRMED_FALSE_RECOVERABLE")
    jobs = jobs[1:]
for rec in pooled_map_deadline(attack_worker, jobs, n_workers=2,
                               stall_timeout_s=float(CYCATTACK_CFG.get("stall_timeout_s", 5400)),
                               meta=meta):
    with open(VERDICT_PATH, "a") as f:
        f.write(dump_line(rec) + "\n")
    if rec.get("verdict") == "CONFIRMED_FALSE_RECOVERABLE":
        kills += 1
        print(f"*** CONFIRMED FALSE RECOVERABLE: {rec['instance_id']} "
              f"via {rec.get('confirming_route')} ***", flush=True)
    if meta["completed"] % 10 == 0:
        el = time.time() - t0
        per = el / max(meta["completed"], 1)
        print(f"[{meta['completed']}/{len(jobs)}] {el / 60:.1f} min "
              f"({per:.0f}s/row, kills={kills})", flush=True)
verdicts = []
if VERDICT_PATH.exists():
    for line in VERDICT_PATH.read_text().splitlines():
        try:
            v = json.loads(line)
        except Exception:
            continue
        verdicts.append(v.get("verdict"))
summary = {
    "shard": SHARD_IDX,
    "n_verdicts_on_file": len(verdicts),
    "kills_total_on_file": sum(1 for v in verdicts
                               if v == "CONFIRMED_FALSE_RECOVERABLE"),
    "wall_h": round((time.time() - T_START) / 3600, 2),
    "note": "these are sheaf-RECOVERABLE assertions on engine-undecided "
            "FORCED-CYCLIC rows; any kill feeds D2-style accounting and "
            "WP3.0c labels",
}
(OUT_DIR / f"cycattack_summary_shard{SHARD_IDX:02d}.json").write_text(
    json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1))
print(f"CYCATTACK SHARD {SHARD_IDX} DONE")


In [ ]:
output_files = sorted(glob.glob(str(OUT_DIR / '*.jsonl')) +
                      glob.glob(str(OUT_DIR / '*.json')) +
                      glob.glob(str(OUT_DIR / '*.csv')))
for output_file in output_files:
    try:
        from google.colab import files
        files.download(output_file)
        print('Downloaded:', output_file)
    except Exception as e:
        print('(Not on Colab / download skipped):', e)
